In [1]:
# Run once if needed.
# In a stable environment, prefer doing this in terminal, then restarting Jupyter.

!pip install -U openai anthropic google-genai pandas tqdm python-dotenv

     |████████████████████████████████| 1.2 MB 4.2 MB/s            
  Attempting uninstall: openai
    Found existing installation: openai 2.32.0
    Uninstalling openai-2.32.0:
      Successfully uninstalled openai-2.32.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llama-index-llms-openai 0.3.38 requires openai<2.0.0,>=1.66.3, but you have openai 2.33.0 which is incompatible.


In [2]:
import os

os.environ["GEMINI_API_KEY"] = ""
os.environ["ANTHROPIC_API_KEY"] = ""
os.environ["OPENAI_API_KEY"] = ""


In [3]:
from __future__ import annotations

import os
import re
import json
import time
import hashlib
import itertools
import datetime as dt
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, Dict, Any, List

import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv

load_dotenv()

TASK_FAMILY = "aut"

BASE_DIR = Path(".")
AI_DATA_DIR = Path("ai_data/aut_generations")
OPENAI_BATCH_DIR = Path("ai_data/aut_batches/openai")
ANTHROPIC_BATCH_DIR = Path("ai_data/aut_batches/anthropic")
GEMINI_BATCH_DIR = Path("ai_data/aut_batches/gemini")
CLEAN_AI_DIR = Path("ai_data/processed")

for p in [AI_DATA_DIR, OPENAI_BATCH_DIR, ANTHROPIC_BATCH_DIR, GEMINI_BATCH_DIR, CLEAN_AI_DIR]:
    p.mkdir(parents=True, exist_ok=True)

In [16]:
from openai import OpenAI
import anthropic
from google import genai
from google.genai import types

# assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY"
# assert os.getenv("ANTHROPIC_API_KEY"), "Missing ANTHROPIC_API_KEY"
# assert os.getenv("GEMINI_API_KEY"), "Missing GEMINI_API_KEY"

openai_client = OpenAI(api_key="")
anthropic_client = anthropic.Anthropic(api_key="")
gemini_client = genai.Client(api_key="")

In [5]:
AUT_OBJECTS = [
    {
        "condition_id": "shoe",
        "condition_label": "Shoe",
        "object": "shoe",
        "common_use": "used as footwear",
    },
    {
        "condition_id": "button",
        "condition_label": "Button",
        "object": "button",
        "common_use": "used to fasten things",
    },
    {
        "condition_id": "key",
        "condition_label": "Key",
        "object": "key",
        "common_use": "used to open a lock",
    },
    {
        "condition_id": "wooden_pencil",
        "condition_label": "Wooden pencil",
        "object": "wooden pencil",
        "common_use": "used for writing",
    },
    {
        "condition_id": "automobile_tire",
        "condition_label": "Automobile tire",
        "object": "automobile tire",
        "common_use": "used on the wheel of an automobile",
    },
]

aut_conditions_df = pd.DataFrame(AUT_OBJECTS)
aut_conditions_df

,condition_id,condition_label,object,common_use
0,shoe,Shoe,shoe,used as footwear
1,button,Button,button,used to fasten things
2,key,Key,key,used to open a lock
3,wooden_pencil,Wooden pencil,wooden pencil,used for writing
4,automobile_tire,Automobile tire,automobile tire,used on the wheel of an automobile


In [6]:
def generate_all_personas():
    trait_pairs = [
        ("extroverted", "introverted"),
        ("agreeable", "antagonistic"),
        ("conscientious", "unconscientious"),
        ("neurotic", "emotionally_stable"),
        ("open to experience", "closed to experience"),
    ]
    return list(itertools.product(*trait_pairs))


def persona_tuple_to_id(persona_tuple) -> str:
    return "__".join(
        t.replace(" ", "_").replace("-", "_")
        for t in persona_tuple
    )


def persona_tuple_to_prompt_text(persona_tuple) -> str:
    traits = ", ".join(persona_tuple)
    return (
        "Write as if you are a person with the following personality profile: "
        f"{traits}. Let this personality influence the creative choice, style, "
        "and framing of the idea, while still following the task exactly."
    )


ALL_PERSONAS = generate_all_personas()

len(ALL_PERSONAS), ALL_PERSONAS[:3]

(32,
 [('extroverted',
   'agreeable',
   'conscientious',
   'neurotic',
   'open to experience'),
  ('extroverted',
   'agreeable',
   'conscientious',
   'neurotic',
   'closed to experience'),
  ('extroverted',
   'agreeable',
   'conscientious',
   'emotionally_stable',
   'open to experience')])

In [7]:
AUT_NEUTRAL_SYSTEM_INSTRUCTIONS = (
    "You are participating in a creativity test. "
    "Follow the task instructions exactly. "
    "Do not explain your reasoning. "
    "Do not include commentary before or after the answer."
)


def build_aut_system_instructions(
    condition_type: str,
    persona_tuple: Optional[tuple] = None,
) -> str:
    """
    condition_type:
        - neutral
        - personality
    """
    if condition_type == "neutral":
        return AUT_NEUTRAL_SYSTEM_INSTRUCTIONS

    if condition_type == "personality":
        if persona_tuple is None:
            raise ValueError("persona_tuple must be provided for personality condition.")

        return (
            AUT_NEUTRAL_SYSTEM_INSTRUCTIONS
            + "\n\n"
            + persona_tuple_to_prompt_text(persona_tuple)
        )

    raise ValueError(f"Unknown condition_type: {condition_type}")


def build_aut_user_prompt(obj: str, common_use: str) -> str:
    return (
        f"Object: {obj}\n"
        f"Common use to avoid: {common_use}\n\n"
        "Generate exactly one unusual, creative, and plausible alternative use "
        "for the object or one of its parts. Do not use the common use. "
        "Do not list multiple uses. Return only the alternative use as a short "
        "phrase or one sentence."
    )


for _, row in aut_conditions_df.iterrows():
    print("=" * 80)
    print(build_aut_user_prompt(row["object"], row["common_use"]))

Object: shoe
Common use to avoid: used as footwear

Generate exactly one unusual, creative, and plausible alternative use for the object or one of its parts. Do not use the common use. Do not list multiple uses. Return only the alternative use as a short phrase or one sentence.
Object: button
Common use to avoid: used to fasten things

Generate exactly one unusual, creative, and plausible alternative use for the object or one of its parts. Do not use the common use. Do not list multiple uses. Return only the alternative use as a short phrase or one sentence.
Object: key
Common use to avoid: used to open a lock

Generate exactly one unusual, creative, and plausible alternative use for the object or one of its parts. Do not use the common use. Do not list multiple uses. Return only the alternative use as a short phrase or one sentence.
Object: wooden pencil
Common use to avoid: used for writing

Generate exactly one unusual, creative, and plausible alternative use for the object or one o

In [8]:
def now_iso() -> str:
    return dt.datetime.now(dt.timezone.utc).isoformat()


def stable_hash(obj: Any) -> str:
    payload = json.dumps(obj, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def append_jsonl(path: Path, record: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def read_jsonl(path: str | Path) -> List[Dict[str, Any]]:
    path = Path(path)
    if not path.exists():
        return []
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records


def load_generation_jsonl(path: str | Path) -> pd.DataFrame:
    return pd.DataFrame(read_jsonl(Path(path)))


def successful_request_keys(path: Path) -> set:
    records = read_jsonl(path)
    return {
        r["request_key"]
        for r in records
        if r.get("status") == "success" and r.get("text")
    }


def make_request_key(
    provider: str,
    model: str,
    task_family: str,
    scenario_name: str,
    condition_id: str,
    condition_type: str,
    temperature: float,
    run_idx: int,
    persona_id: Optional[str],
    system_instructions: str,
    user_prompt: str,
) -> str:
    return stable_hash(
        {
            "provider": provider,
            "model": model,
            "task_family": task_family,
            "scenario_name": scenario_name,
            "condition_id": condition_id,
            "condition_type": condition_type,
            "temperature": float(temperature),
            "run_idx": int(run_idx),
            "persona_id": persona_id,
            "system_instructions": system_instructions,
            "user_prompt": user_prompt,
        }
    )

In [9]:
@dataclass
class GenerationScenario:
    scenario_name: str
    condition_type: str
    temperatures: List[float]
    n_per_condition: int
    condition_ids: List[str]
    include_personas: bool = False
    max_output_tokens: int = 120
    description: str = ""


@dataclass
class ProviderModelConfig:
    provider: str
    model: str
    batch_dir: Path
    temperature_grid: List[float]
    neutral_temperature_robustness_grid: List[float]
    personality_temperature_grid: List[float]
    max_output_tokens: int = 120


MODEL_CONFIGS = {
    "openai_gpt54": ProviderModelConfig(
        provider="openai",
        model="gpt-5.4",
        batch_dir=OPENAI_BATCH_DIR,
        temperature_grid=[0.7, 1.0, 1.3],
        neutral_temperature_robustness_grid=[0.7, 1.3],
        personality_temperature_grid=[0.7, 1.0, 1.3],
        max_output_tokens=120,
    ),
    "claude_sonnet45": ProviderModelConfig(
        provider="anthropic",
        model="claude-sonnet-4-5",
        batch_dir=ANTHROPIC_BATCH_DIR,
        temperature_grid=[0.3, 0.7, 1.0],
        neutral_temperature_robustness_grid=[0.3, 0.7],
        personality_temperature_grid=[0.3, 0.7, 1.0],
        max_output_tokens=120,
    ),
    "gemini_flash25": ProviderModelConfig(
        provider="gemini",
        model="gemini-2.5-flash",
        batch_dir=GEMINI_BATCH_DIR,
        temperature_grid=[0.7, 1.0, 1.3],
        neutral_temperature_robustness_grid=[0.7, 1.3],
        personality_temperature_grid=[0.7, 1.0, 1.3],
        max_output_tokens=120,
    ),
}


def build_scenarios_for_provider(config: ProviderModelConfig) -> Dict[str, GenerationScenario]:
    condition_ids = aut_conditions_df["condition_id"].tolist()

    return {
        "neutral_main_t1": GenerationScenario(
            scenario_name="neutral_main_t1",
            condition_type="neutral",
            temperatures=[1.0],
            n_per_condition=50,
            condition_ids=condition_ids,
            include_personas=False,
            max_output_tokens=config.max_output_tokens,
            description="Primary AUT benchmark: neutral prompt, temperature 1.0, 50 generations per object.",
        ),
        "neutral_temperature_robustness": GenerationScenario(
            scenario_name="neutral_temperature_robustness",
            condition_type="neutral",
            temperatures=config.neutral_temperature_robustness_grid,
            n_per_condition=10,
            condition_ids=condition_ids,
            include_personas=False,
            max_output_tokens=config.max_output_tokens,
            description="AUT temperature robustness: neutral prompt at non-default temperatures.",
        ),
        "personality_grid": GenerationScenario(
            scenario_name="personality_grid",
            condition_type="personality",
            temperatures=config.personality_temperature_grid,
            n_per_condition=10,
            condition_ids=condition_ids,
            include_personas=True,
            max_output_tokens=config.max_output_tokens,
            description="AUT personality robustness: 32 Big-Five-style profiles crossed with temperature.",
        ),
    }


for name, cfg in MODEL_CONFIGS.items():
    sc = build_scenarios_for_provider(cfg)
    print(name, {k: (v.temperatures, v.n_per_condition, v.include_personas) for k, v in sc.items()})

openai_gpt54 {'neutral_main_t1': ([1.0], 50, False), 'neutral_temperature_robustness': ([0.7, 1.3], 10, False), 'personality_grid': ([0.7, 1.0, 1.3], 10, True)}
claude_sonnet45 {'neutral_main_t1': ([1.0], 50, False), 'neutral_temperature_robustness': ([0.3, 0.7], 10, False), 'personality_grid': ([0.3, 0.7, 1.0], 10, True)}
gemini_flash25 {'neutral_main_t1': ([1.0], 50, False), 'neutral_temperature_robustness': ([0.7, 1.3], 10, False), 'personality_grid': ([0.7, 1.0, 1.3], 10, True)}


In [10]:
def expected_request_count(config: ProviderModelConfig) -> pd.DataFrame:
    scenarios = build_scenarios_for_provider(config)
    rows = []
    for name, s in scenarios.items():
        n_personas = len(ALL_PERSONAS) if s.include_personas else 1
        rows.append({
            "scenario_name": name,
            "n_conditions": len(s.condition_ids),
            "n_temperatures": len(s.temperatures),
            "n_personas": n_personas,
            "n_per_condition": s.n_per_condition,
            "n_requests": len(s.condition_ids) * len(s.temperatures) * n_personas * s.n_per_condition,
        })
    return pd.DataFrame(rows)

for name, cfg in MODEL_CONFIGS.items():
    print("=" * 80)
    print(name)
    display(expected_request_count(cfg))

openai_gpt54


,scenario_name,n_conditions,n_temperatures,n_personas,n_per_condition,n_requests
0,neutral_main_t1,5,1,1,50,250
1,neutral_temperature_robustness,5,2,1,10,100
2,personality_grid,5,3,32,10,4800


claude_sonnet45


,scenario_name,n_conditions,n_temperatures,n_personas,n_per_condition,n_requests
0,neutral_main_t1,5,1,1,50,250
1,neutral_temperature_robustness,5,2,1,10,100
2,personality_grid,5,3,32,10,4800


gemini_flash25


,scenario_name,n_conditions,n_temperatures,n_personas,n_per_condition,n_requests
0,neutral_main_t1,5,1,1,50,250
1,neutral_temperature_robustness,5,2,1,10,100
2,personality_grid,5,3,32,10,4800


In [11]:
def build_aut_generation_plan(
    scenario: GenerationScenario,
    conditions_df: pd.DataFrame,
    provider_name: str,
    model_name: str,
) -> pd.DataFrame:
    rows = []

    conditions_use = (
        conditions_df[conditions_df["condition_id"].isin(scenario.condition_ids)]
        .copy()
        .sort_values("condition_id")
    )

    persona_tuples = ALL_PERSONAS if scenario.include_personas else [None]

    for _, condition_row in conditions_use.iterrows():
        condition_id = condition_row["condition_id"]
        condition_label = condition_row["condition_label"]
        obj = condition_row["object"]
        common_use = condition_row["common_use"]
        user_prompt = build_aut_user_prompt(obj=obj, common_use=common_use)

        for temperature in scenario.temperatures:
            for persona_tuple in persona_tuples:
                if persona_tuple is None:
                    persona_id = None
                    persona_traits = None
                else:
                    persona_id = persona_tuple_to_id(persona_tuple)
                    persona_traits = list(persona_tuple)

                system_instructions = build_aut_system_instructions(
                    condition_type=scenario.condition_type,
                    persona_tuple=persona_tuple,
                )

                for run_idx in range(scenario.n_per_condition):
                    request_key = make_request_key(
                        provider=provider_name,
                        model=model_name,
                        task_family=TASK_FAMILY,
                        scenario_name=scenario.scenario_name,
                        condition_id=condition_id,
                        condition_type=scenario.condition_type,
                        temperature=float(temperature),
                        run_idx=int(run_idx),
                        persona_id=persona_id,
                        system_instructions=system_instructions,
                        user_prompt=user_prompt,
                    )

                    rows.append({
                        "request_key": request_key,
                        "task_family": TASK_FAMILY,
                        "scenario_name": scenario.scenario_name,
                        "provider": provider_name,
                        "model": model_name,
                        "condition_type": scenario.condition_type,
                        "condition_id": condition_id,
                        "condition_label": condition_label,
                        "object": obj,
                        "common_use": common_use,
                        "temperature": float(temperature),
                        "run_idx": int(run_idx),
                        "persona_id": persona_id,
                        "persona_traits": persona_traits,
                        "system_instructions": system_instructions,
                        "user_prompt": user_prompt,
                        "max_output_tokens": int(scenario.max_output_tokens),
                    })

    return pd.DataFrame(rows)


# Smoke-check plan shape for one provider.
cfg = MODEL_CONFIGS["openai_gpt54"]
scenarios = build_scenarios_for_provider(cfg)

test_plan = build_aut_generation_plan(
    scenario=scenarios["neutral_main_t1"],
    conditions_df=aut_conditions_df,
    provider_name=cfg.provider,
    model_name=cfg.model,
)

test_plan.shape, test_plan.head()

((250, 17),
                                          request_key task_family  \
 0  532719c7aa6d48a68b9d1cd60fd9ee626380b5f9d5db27...         aut   
 1  c9bdf30c3284556b758c55c7770c2c804d4719d0b38c12...         aut   
 2  9a61b021440787c5cf518c4420c160a73735257f1edf42...         aut   
 3  217d225ea872f97f735f81f9b786630b37c475be61d857...         aut   
 4  747db15c24d6a60cbee9e145e9542d9c8b8692cca5740b...         aut   
 
      scenario_name provider    model condition_type     condition_id  \
 0  neutral_main_t1   openai  gpt-5.4        neutral  automobile_tire   
 1  neutral_main_t1   openai  gpt-5.4        neutral  automobile_tire   
 2  neutral_main_t1   openai  gpt-5.4        neutral  automobile_tire   
 3  neutral_main_t1   openai  gpt-5.4        neutral  automobile_tire   
 4  neutral_main_t1   openai  gpt-5.4        neutral  automobile_tire   
 
    condition_label           object                          common_use  \
 0  Automobile tire  automobile tire  used on the wheel 

In [12]:
def load_all_generation_files(
    provider_name: str,
    model_name: str,
    output_dir: Path = AI_DATA_DIR,
) -> pd.DataFrame:
    paths = sorted(output_dir.glob(f"{provider_name}__{model_name}__*.jsonl"))
    paths = [p for p in paths if "__plan" not in p.name]

    dfs = []
    for p in paths:
        df = load_generation_jsonl(p)
        if len(df) > 0:
            df["source_file"] = str(p)
            dfs.append(df)

    if not dfs:
        return pd.DataFrame()

    return pd.concat(dfs, ignore_index=True)


def deduplicate_generation_records(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df

    df = df.copy()
    status_priority = {"success": 3, "empty_text": 2, "error": 1}
    df["_status_priority"] = df["status"].map(status_priority).fillna(0)
    df["_original_order"] = range(len(df))

    df = (
        df.sort_values(["request_key", "_status_priority", "_original_order"])
        .drop_duplicates("request_key", keep="last")
        .drop(columns=["_status_priority", "_original_order"])
        .reset_index(drop=True)
    )

    return df


def add_aut_validity_diagnostics(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    def word_count(x):
        if not isinstance(x, str):
            return 0
        return len(re.findall(r"\b[\w']+\b", x))

    def line_count(x):
        if not isinstance(x, str):
            return 0
        return len([ln for ln in x.splitlines() if ln.strip()])

    def looks_like_list(x):
        if not isinstance(x, str):
            return False
        lines = [ln.strip() for ln in x.splitlines() if ln.strip()]
        if len(lines) > 1:
            return True
        return bool(re.search(r"(^|\n)\s*(\d+[\.\)]|[-*•])\s+", x))

    def likely_multiple_uses(x):
        if not isinstance(x, str):
            return False
        lower = x.lower()
        if looks_like_list(x):
            return True
        # Heuristic: semicolon-separated or "or" alternatives often indicate multiple uses.
        if ";" in x:
            return True
        if re.search(r"\b(or|and also|another use|alternatively)\b", lower):
            return True
        return False

    common_use_terms = {
        "shoe": ["wear", "footwear", "feet", "foot", "walking", "walk", "running", "run"],
        "button": ["fasten", "fastening", "buttoning", "clothes", "shirt"],
        "key": ["open a lock", "unlock", "lock", "door"],
        "wooden_pencil": ["write", "writing", "draw", "drawing"],
        "automobile_tire": ["wheel", "car", "automobile", "drive", "driving"],
    }

    def repeats_primary_use(row):
        text = row.get("text")
        if not isinstance(text, str):
            return False
        lower = text.lower()
        terms = common_use_terms.get(row.get("condition_id"), [])
        return any(term in lower for term in terms)

    df["output_word_count"] = df["text"].map(word_count)
    df["output_line_count"] = df["text"].map(line_count)
    df["looks_like_list"] = df["text"].map(looks_like_list)
    df["likely_multiple_uses"] = df["text"].map(likely_multiple_uses)
    df["repeats_primary_use_terms"] = df.apply(repeats_primary_use, axis=1)

    df["valid_exactly_one_short_response_heuristic"] = (
        (df["status"] == "success")
        & (df["output_word_count"].between(2, 30))
        & (~df["likely_multiple_uses"])
    )

    return df


def save_provider_outputs(provider: str, model: str, df: pd.DataFrame):
    audit_pkl = CLEAN_AI_DIR / f"{provider}__{model}__aut_generations_all_records_with_errors.pkl"
    audit_csv = CLEAN_AI_DIR / f"{provider}__{model}__aut_generations_all_records_with_errors.csv"
    success_pkl = CLEAN_AI_DIR / f"{provider}__{model}__aut_generations_success_only.pkl"
    success_csv = CLEAN_AI_DIR / f"{provider}__{model}__aut_generations_success_only.csv"

    df = add_aut_validity_diagnostics(df)
    success_df = df.query("status == 'success'").copy()

    df.to_pickle(audit_pkl)
    df.to_csv(audit_csv, index=False)
    success_df.to_pickle(success_pkl)
    success_df.to_csv(success_csv, index=False)

    print("Saved:")
    print(audit_pkl)
    print(audit_csv)
    print(success_pkl)
    print(success_csv)

    return {
        "audit_pkl": audit_pkl,
        "audit_csv": audit_csv,
        "success_pkl": success_pkl,
        "success_csv": success_csv,
    }


def summarize_provider_outputs(provider: str, model: str) -> pd.DataFrame:
    df = load_all_generation_files(provider, model, AI_DATA_DIR)
    df = deduplicate_generation_records(df)
    if df.empty:
        return pd.DataFrame()

    df = add_aut_validity_diagnostics(df)

    summary = (
        df.groupby(["scenario_name", "condition_id", "temperature", "status"], dropna=False)
        .size()
        .reset_index(name="n")
        .sort_values(["scenario_name", "condition_id", "temperature", "status"])
    )

    display(summary)
    return df

# OpenAI

In [13]:
def make_openai_batch_input_jsonl(
    scenario_name: str,
    config: ProviderModelConfig,
    conditions_df: pd.DataFrame,
    output_dir: Path = OPENAI_BATCH_DIR,
    exclude_existing_successes: bool = True,
) -> tuple[Path, Path, pd.DataFrame]:
    scenarios = build_scenarios_for_provider(config)
    scenario = scenarios[scenario_name]

    plan_df = build_aut_generation_plan(
        scenario=scenario,
        conditions_df=conditions_df,
        provider_name=config.provider,
        model_name=config.model,
    )

    existing_output_path = AI_DATA_DIR / f"{config.provider}__{config.model}__{scenario_name}.jsonl"
    if exclude_existing_successes and existing_output_path.exists():
        done_keys = successful_request_keys(existing_output_path)
        plan_df = plan_df[~plan_df["request_key"].isin(done_keys)].copy()

    timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    stem = f"{config.provider}__{config.model}__{TASK_FAMILY}__{scenario_name}__{timestamp}"

    batch_jsonl_path = output_dir / f"{stem}__batch_input.jsonl"
    plan_csv_path = output_dir / f"{stem}__batch_plan.csv"

    plan_df.to_csv(plan_csv_path, index=False)

    with open(batch_jsonl_path, "w", encoding="utf-8") as f:
        for _, row in plan_df.iterrows():
            body = {
                "model": config.model,
                "instructions": row["system_instructions"],
                "input": row["user_prompt"],
                "temperature": float(row["temperature"]),
                "max_output_tokens": int(row["max_output_tokens"]),
            }

            request = {
                "custom_id": row["request_key"],
                "method": "POST",
                "url": "/v1/responses",
                "body": body,
            }

            f.write(json.dumps(request, ensure_ascii=False) + "\n")

    print(f"OpenAI scenario: {scenario_name}")
    print(f"Requests written: {len(plan_df):,}")
    print(f"Batch JSONL: {batch_jsonl_path}")
    print(f"Plan CSV:    {plan_csv_path}")

    return batch_jsonl_path, plan_csv_path, plan_df


def submit_openai_batch(batch_jsonl_path: Path, scenario_name: str, model_name: str) -> dict:
    batch_input_file = openai_client.files.create(
        file=open(batch_jsonl_path, "rb"),
        purpose="batch",
    )

    batch = openai_client.batches.create(
        input_file_id=batch_input_file.id,
        endpoint="/v1/responses",
        completion_window="24h",
        metadata={
            "task_family": TASK_FAMILY,
            "scenario_name": scenario_name,
            "model": model_name,
            "local_input_file": str(batch_jsonl_path),
        },
    )

    batch_info = {
        "batch_id": batch.id,
        "input_file_id": batch_input_file.id,
        "scenario_name": scenario_name,
        "model": model_name,
        "submitted_at_utc": now_iso(),
        "local_input_file": str(batch_jsonl_path),
    }

    manifest_path = batch_jsonl_path.with_suffix(".batch_manifest.json")
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(batch_info, f, indent=2)

    print("Submitted OpenAI batch:")
    print(json.dumps(batch_info, indent=2))

    return batch_info


def check_openai_batch(batch_id: str):
    batch = openai_client.batches.retrieve(batch_id)
    print(batch)
    return batch


def download_openai_batch_results(batch_id: str, output_dir: Path = OPENAI_BATCH_DIR):
    batch = openai_client.batches.retrieve(batch_id)

    if batch.status != "completed":
        print(f"Batch is not completed yet. Current status: {batch.status}")
        return None, None

    output_path = output_dir / f"{batch_id}__output.jsonl"
    error_path = output_dir / f"{batch_id}__errors.jsonl"

    if batch.output_file_id:
        file_response = openai_client.files.content(batch.output_file_id)
        output_path.write_text(file_response.text, encoding="utf-8")
        print(f"Downloaded output: {output_path}")

    if batch.error_file_id:
        error_response = openai_client.files.content(batch.error_file_id)
        error_path.write_text(error_response.text, encoding="utf-8")
        print(f"Downloaded errors: {error_path}")
    else:
        error_path = None

    return output_path, error_path


def extract_text_from_responses_api_body(body: dict) -> str:
    if not isinstance(body, dict):
        return ""

    if body.get("output_text"):
        return str(body["output_text"]).strip()

    texts = []
    for item in body.get("output", []) or []:
        for content in item.get("content", []) or []:
            if content.get("type") in {"output_text", "text"} and "text" in content:
                texts.append(content["text"])

    return "\n".join(texts).strip()


def parse_openai_batch_output_to_standard_jsonl(
    batch_output_path: Path,
    plan_csv_path: Path,
    scenario_name: str,
    config: ProviderModelConfig,
    standard_output_dir: Path = AI_DATA_DIR,
) -> Path:
    plan_df = pd.read_csv(plan_csv_path)
    plan_by_key = {
        row["request_key"]: row.to_dict()
        for _, row in plan_df.iterrows()
    }

    standard_output_path = standard_output_dir / f"{config.provider}__{config.model}__{scenario_name}.jsonl"
    batch_records = read_jsonl(batch_output_path)

    n_success = 0
    n_error = 0

    for rec in batch_records:
        request_key = rec.get("custom_id")
        plan_row = plan_by_key.get(request_key, {})
        response = rec.get("response")
        error = rec.get("error")

        if response and response.get("body"):
            body = response["body"]
            text = extract_text_from_responses_api_body(body)
            usage = body.get("usage")

            record = {
                **plan_row,
                "created_at_utc": now_iso(),
                "status": "success" if text else "empty_text",
                "text": text if text else None,
                "provider_response_id": body.get("id"),
                "usage": usage,
                "raw_response": None,
                "error": None if text else "No text extracted from response body.",
                "batch_custom_id": request_key,
                "batch_output_file": str(batch_output_path),
            }

            n_success += int(bool(text))
            n_error += int(not bool(text))
        else:
            record = {
                **plan_row,
                "created_at_utc": now_iso(),
                "status": "error",
                "text": None,
                "provider_response_id": None,
                "usage": None,
                "raw_response": None,
                "error": error,
                "batch_custom_id": request_key,
                "batch_output_file": str(batch_output_path),
            }
            n_error += 1

        append_jsonl(standard_output_path, record)

    print(f"Parsed OpenAI batch output to: {standard_output_path}")
    print(f"Success-ish text records: {n_success:,}")
    print(f"Errors/empty:             {n_error:,}")

    return standard_output_path

In [14]:
def submit_openai_scenario(scenario_name: str, exclude_existing_successes: bool = True) -> dict:
    cfg = MODEL_CONFIGS["openai_gpt54"]

    batch_jsonl_path, plan_csv_path, plan_df = make_openai_batch_input_jsonl(
        scenario_name=scenario_name,
        config=cfg,
        conditions_df=aut_conditions_df,
        output_dir=OPENAI_BATCH_DIR,
        exclude_existing_successes=exclude_existing_successes,
    )

    if len(plan_df) == 0:
        print(f"No remaining requests for {scenario_name}.")
        return {
            "scenario_name": scenario_name,
            "model": cfg.model,
            "provider": cfg.provider,
            "n_requests_submitted": 0,
            "plan_csv_path": str(plan_csv_path),
            "batch_jsonl_path": str(batch_jsonl_path),
            "batch_id": None,
        }

    batch_info = submit_openai_batch(
        batch_jsonl_path=batch_jsonl_path,
        scenario_name=scenario_name,
        model_name=cfg.model,
    )

    batch_info["provider"] = cfg.provider
    batch_info["plan_csv_path"] = str(plan_csv_path)
    batch_info["batch_jsonl_path"] = str(batch_jsonl_path)
    batch_info["n_requests_submitted"] = len(plan_df)

    return batch_info


def parse_completed_openai_batch(batch_info: dict):
    cfg = MODEL_CONFIGS["openai_gpt54"]
    batch_id = batch_info["batch_id"]

    output_path, error_path = download_openai_batch_results(batch_id, OPENAI_BATCH_DIR)
    if output_path is None:
        return None

    standard_path = parse_openai_batch_output_to_standard_jsonl(
        batch_output_path=output_path,
        plan_csv_path=Path(batch_info["plan_csv_path"]),
        scenario_name=batch_info["scenario_name"],
        config=cfg,
    )

    return standard_path

In [17]:
openai_main_batch = submit_openai_scenario("neutral_main_t1")
openai_main_batch

OpenAI scenario: neutral_main_t1
Requests written: 250
Batch JSONL: ai_data/aut_batches/openai/openai__gpt-5.4__aut__neutral_main_t1__20260429_110001__batch_input.jsonl
Plan CSV:    ai_data/aut_batches/openai/openai__gpt-5.4__aut__neutral_main_t1__20260429_110001__batch_plan.csv
Submitted OpenAI batch:
{
  "batch_id": "batch_69f21cf40e4481908932f3ce97cbb8e8",
  "input_file_id": "file-BByrya4vjsvAToA7eTMwEx",
  "scenario_name": "neutral_main_t1",
  "model": "gpt-5.4",
  "submitted_at_utc": "2026-04-29T15:00:04.860623+00:00",
  "local_input_file": "ai_data/aut_batches/openai/openai__gpt-5.4__aut__neutral_main_t1__20260429_110001__batch_input.jsonl"
}


{'batch_id': 'batch_69f21cf40e4481908932f3ce97cbb8e8',
 'input_file_id': 'file-BByrya4vjsvAToA7eTMwEx',
 'scenario_name': 'neutral_main_t1',
 'model': 'gpt-5.4',
 'submitted_at_utc': '2026-04-29T15:00:04.860623+00:00',
 'local_input_file': 'ai_data/aut_batches/openai/openai__gpt-5.4__aut__neutral_main_t1__20260429_110001__batch_input.jsonl',
 'provider': 'openai',
 'plan_csv_path': 'ai_data/aut_batches/openai/openai__gpt-5.4__aut__neutral_main_t1__20260429_110001__batch_plan.csv',
 'batch_jsonl_path': 'ai_data/aut_batches/openai/openai__gpt-5.4__aut__neutral_main_t1__20260429_110001__batch_input.jsonl',
 'n_requests_submitted': 250}

In [23]:
check_openai_batch(openai_main_batch["batch_id"])

Batch(id='batch_69f21cf40e4481908932f3ce97cbb8e8', completion_window='24h', created_at=1777474804, endpoint='/v1/responses', input_file_id='file-BByrya4vjsvAToA7eTMwEx', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1777478475, error_file_id=None, errors=None, expired_at=None, expires_at=1777561204, failed_at=None, finalizing_at=1777478459, in_progress_at=1777474865, metadata={'task_family': 'aut', 'scenario_name': 'neutral_main_t1', 'model': 'gpt-5.4', 'local_input_file': 'ai_data/aut_batches/openai/openai__gpt-5.4__aut__neutral_main_t1__20260429_110001__batch_input.jsonl'}, model='gpt-5.4-2026-03-05', output_file_id='file-PHLuLYidNQsz5haXp8qHub', request_counts=BatchRequestCounts(completed=250, failed=0, total=250), usage=BatchUsage(input_tokens=25250, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=5393, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=30643))


Batch(id='batch_69f21cf40e4481908932f3ce97cbb8e8', completion_window='24h', created_at=1777474804, endpoint='/v1/responses', input_file_id='file-BByrya4vjsvAToA7eTMwEx', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1777478475, error_file_id=None, errors=None, expired_at=None, expires_at=1777561204, failed_at=None, finalizing_at=1777478459, in_progress_at=1777474865, metadata={'task_family': 'aut', 'scenario_name': 'neutral_main_t1', 'model': 'gpt-5.4', 'local_input_file': 'ai_data/aut_batches/openai/openai__gpt-5.4__aut__neutral_main_t1__20260429_110001__batch_input.jsonl'}, model='gpt-5.4-2026-03-05', output_file_id='file-PHLuLYidNQsz5haXp8qHub', request_counts=BatchRequestCounts(completed=250, failed=0, total=250), usage=BatchUsage(input_tokens=25250, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=5393, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=30643))

In [24]:
openai_main_standard_path = parse_completed_openai_batch(openai_main_batch)

openai_main_df = load_generation_jsonl(openai_main_standard_path)
print(openai_main_df.shape)
print(openai_main_df["status"].value_counts(dropna=False))
display(openai_main_df[["condition_id", "temperature", "run_idx", "status", "text"]].head(10))

Downloaded output: ai_data/aut_batches/openai/batch_69f21cf40e4481908932f3ce97cbb8e8__output.jsonl
Parsed OpenAI batch output to: ai_data/aut_generations/openai__gpt-5.4__neutral_main_t1.jsonl
Success-ish text records: 250
Errors/empty:             0
(250, 26)
status
success    250
Name: count, dtype: int64


,condition_id,temperature,run_idx,status,text
0,automobile_tire,1.0,0,success,A tire can be buried halfway in the ground to ...
1,automobile_tire,1.0,1,success,Hang it horizontally as a shock-absorbing live...
2,automobile_tire,1.0,2,success,Hang it horizontally as a swinging garden plan...
3,automobile_tire,1.0,3,success,A tire hung horizontally from a sturdy frame c...
4,automobile_tire,1.0,4,success,A hanging herb garden planter made by suspendi...
5,automobile_tire,1.0,5,success,A tire laid flat and filled with soil makes a ...
6,automobile_tire,1.0,6,success,Half-bury it in a playground as a durable clim...
7,automobile_tire,1.0,7,success,A tire hung horizontally from a tree branch ca...
8,automobile_tire,1.0,8,success,Suspend it from a sturdy beam as a cushioned h...
9,automobile_tire,1.0,9,success,Hang it horizontally as a suspended garden pla...


In [25]:
openai_main_df = add_aut_validity_diagnostics(openai_main_df)

openai_main_df.groupby("condition_id").agg(
    n=("text", "size"),
    n_success=("status", lambda x: (x == "success").sum()),
    mean_words=("output_word_count", "mean"),
    median_words=("output_word_count", "median"),
    validity_rate=("valid_exactly_one_short_response_heuristic", "mean"),
)

,n,n_success,mean_words,median_words,validity_rate
condition_id,,,,,
automobile_tire,50,50,13.92,13.0,0.88
button,50,50,15.68,15.0,0.88
key,50,50,17.92,18.0,0.76
shoe,50,50,14.14,14.0,0.72
wooden_pencil,50,50,16.34,17.0,0.98


In [26]:
openai_temp_batch = submit_openai_scenario("neutral_temperature_robustness")
openai_personality_batch = submit_openai_scenario("personality_grid")

openai_batches = pd.DataFrame([openai_main_batch, openai_temp_batch, openai_personality_batch])
openai_manifest_path = OPENAI_BATCH_DIR / "openai__gpt-5.4__aut_batch_manifest.csv"
openai_batches.to_csv(openai_manifest_path, index=False)
openai_batches

OpenAI scenario: neutral_temperature_robustness
Requests written: 100
Batch JSONL: ai_data/aut_batches/openai/openai__gpt-5.4__aut__neutral_temperature_robustness__20260429_120357__batch_input.jsonl
Plan CSV:    ai_data/aut_batches/openai/openai__gpt-5.4__aut__neutral_temperature_robustness__20260429_120357__batch_plan.csv
Submitted OpenAI batch:
{
  "batch_id": "batch_69f22bee96e08190babc2f6e11505b7f",
  "input_file_id": "file-Y5fFkbu72fn4BJ2z1gtakF",
  "scenario_name": "neutral_temperature_robustness",
  "model": "gpt-5.4",
  "submitted_at_utc": "2026-04-29T16:03:59.405595+00:00",
  "local_input_file": "ai_data/aut_batches/openai/openai__gpt-5.4__aut__neutral_temperature_robustness__20260429_120357__batch_input.jsonl"
}
OpenAI scenario: personality_grid
Requests written: 4,800
Batch JSONL: ai_data/aut_batches/openai/openai__gpt-5.4__aut__personality_grid__20260429_120359__batch_input.jsonl
Plan CSV:    ai_data/aut_batches/openai/openai__gpt-5.4__aut__personality_grid__20260429_120359

,batch_id,input_file_id,scenario_name,model,submitted_at_utc,local_input_file,provider,plan_csv_path,batch_jsonl_path,n_requests_submitted
0,batch_69f21cf40e4481908932f3ce97cbb8e8,file-BByrya4vjsvAToA7eTMwEx,neutral_main_t1,gpt-5.4,2026-04-29T15:00:04.860623+00:00,ai_data/aut_batches/openai/openai__gpt-5.4__au...,openai,ai_data/aut_batches/openai/openai__gpt-5.4__au...,ai_data/aut_batches/openai/openai__gpt-5.4__au...,250
1,batch_69f22bee96e08190babc2f6e11505b7f,file-Y5fFkbu72fn4BJ2z1gtakF,neutral_temperature_robustness,gpt-5.4,2026-04-29T16:03:59.405595+00:00,ai_data/aut_batches/openai/openai__gpt-5.4__au...,openai,ai_data/aut_batches/openai/openai__gpt-5.4__au...,ai_data/aut_batches/openai/openai__gpt-5.4__au...,100
2,batch_69f22bf131088190a37c091663450f08,file-SbJx8fQ6nkJ5BskMSZRuHu,personality_grid,gpt-5.4,2026-04-29T16:04:01.370736+00:00,ai_data/aut_batches/openai/openai__gpt-5.4__au...,openai,ai_data/aut_batches/openai/openai__gpt-5.4__au...,ai_data/aut_batches/openai/openai__gpt-5.4__au...,4800


In [33]:
for _, row in openai_batches.dropna(subset=["batch_id"]).iterrows():
    print("=" * 100)
    print(row["scenario_name"])
    check_openai_batch(row["batch_id"])

neutral_main_t1
Batch(id='batch_69f21cf40e4481908932f3ce97cbb8e8', completion_window='24h', created_at=1777474804, endpoint='/v1/responses', input_file_id='file-BByrya4vjsvAToA7eTMwEx', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1777478475, error_file_id=None, errors=None, expired_at=None, expires_at=1777561204, failed_at=None, finalizing_at=1777478459, in_progress_at=1777474865, metadata={'task_family': 'aut', 'scenario_name': 'neutral_main_t1', 'model': 'gpt-5.4', 'local_input_file': 'ai_data/aut_batches/openai/openai__gpt-5.4__aut__neutral_main_t1__20260429_110001__batch_input.jsonl'}, model='gpt-5.4-2026-03-05', output_file_id='file-PHLuLYidNQsz5haXp8qHub', request_counts=BatchRequestCounts(completed=250, failed=0, total=250), usage=BatchUsage(input_tokens=25250, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=5393, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=30643))
neutral_tempera

In [34]:
for batch_info in [openai_temp_batch, openai_personality_batch]:
    parse_completed_openai_batch(batch_info)

Downloaded output: ai_data/aut_batches/openai/batch_69f22bee96e08190babc2f6e11505b7f__output.jsonl
Parsed OpenAI batch output to: ai_data/aut_generations/openai__gpt-5.4__neutral_temperature_robustness.jsonl
Success-ish text records: 100
Errors/empty:             0
Downloaded output: ai_data/aut_batches/openai/batch_69f22bf131088190a37c091663450f08__output.jsonl
Parsed OpenAI batch output to: ai_data/aut_generations/openai__gpt-5.4__personality_grid.jsonl
Success-ish text records: 4,800
Errors/empty:             0


In [35]:
openai_all_df = load_all_generation_files("openai", "gpt-5.4", AI_DATA_DIR)
openai_all_df = deduplicate_generation_records(openai_all_df)
openai_all_df = add_aut_validity_diagnostics(openai_all_df)

print(openai_all_df["scenario_name"].value_counts(dropna=False))
print(openai_all_df["status"].value_counts(dropna=False))

save_provider_outputs("openai", "gpt-5.4", openai_all_df)

scenario_name
personality_grid                  4800
neutral_main_t1                    250
neutral_temperature_robustness     100
Name: count, dtype: int64
status
success    5150
Name: count, dtype: int64
Saved:
ai_data/processed/openai__gpt-5.4__aut_generations_all_records_with_errors.pkl
ai_data/processed/openai__gpt-5.4__aut_generations_all_records_with_errors.csv
ai_data/processed/openai__gpt-5.4__aut_generations_success_only.pkl
ai_data/processed/openai__gpt-5.4__aut_generations_success_only.csv


{'audit_pkl': PosixPath('ai_data/processed/openai__gpt-5.4__aut_generations_all_records_with_errors.pkl'),
 'audit_csv': PosixPath('ai_data/processed/openai__gpt-5.4__aut_generations_all_records_with_errors.csv'),
 'success_pkl': PosixPath('ai_data/processed/openai__gpt-5.4__aut_generations_success_only.pkl'),
 'success_csv': PosixPath('ai_data/processed/openai__gpt-5.4__aut_generations_success_only.csv')}

# Claude

In [36]:
def make_anthropic_custom_id(request_key: str) -> str:
    return "r_" + request_key[:48]


def build_anthropic_generation_plan(
    scenario_name: str,
    config: ProviderModelConfig,
    conditions_df: pd.DataFrame,
) -> pd.DataFrame:
    scenarios = build_scenarios_for_provider(config)
    scenario = scenarios[scenario_name]

    plan_df = build_aut_generation_plan(
        scenario=scenario,
        conditions_df=conditions_df,
        provider_name=config.provider,
        model_name=config.model,
    ).copy()

    plan_df["anthropic_custom_id"] = plan_df["request_key"].map(make_anthropic_custom_id)

    if plan_df["anthropic_custom_id"].duplicated().any():
        raise ValueError("Duplicate anthropic_custom_id found. Increase hash length.")

    return plan_df


def make_anthropic_batch_requests(plan_df: pd.DataFrame) -> list:
    requests = []

    for _, row in plan_df.iterrows():
        params = {
            "model": row["model"],
            "max_tokens": int(row["max_output_tokens"]),
            "temperature": float(row["temperature"]),
            "system": row["system_instructions"],
            "messages": [
                {
                    "role": "user",
                    "content": row["user_prompt"],
                }
            ],
        }

        requests.append({
            "custom_id": row["anthropic_custom_id"],
            "params": params,
        })

    return requests


def submit_anthropic_batch(
    scenario_name: str,
    config: ProviderModelConfig,
    conditions_df: pd.DataFrame,
    output_dir: Path = ANTHROPIC_BATCH_DIR,
    exclude_existing_successes: bool = True,
) -> dict:
    standard_output_path = AI_DATA_DIR / f"{config.provider}__{config.model}__{scenario_name}.jsonl"

    plan_df = build_anthropic_generation_plan(
        scenario_name=scenario_name,
        config=config,
        conditions_df=conditions_df,
    )

    if exclude_existing_successes and standard_output_path.exists():
        done_keys = successful_request_keys(standard_output_path)
        plan_df = plan_df[~plan_df["request_key"].isin(done_keys)].copy()

    timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    stem = f"{config.provider}__{config.model}__{TASK_FAMILY}__{scenario_name}__{timestamp}"

    plan_csv_path = output_dir / f"{stem}__batch_plan.csv"
    request_json_path = output_dir / f"{stem}__batch_requests.json"

    plan_df.to_csv(plan_csv_path, index=False)

    requests = make_anthropic_batch_requests(plan_df)

    with open(request_json_path, "w", encoding="utf-8") as f:
        json.dump(requests, f, indent=2, ensure_ascii=False)

    print(f"Anthropic scenario: {scenario_name}")
    print(f"Requests to submit: {len(requests):,}")
    print(f"Plan CSV: {plan_csv_path}")
    print(f"Request JSON: {request_json_path}")

    if len(requests) == 0:
        return {
            "batch_id": None,
            "scenario_name": scenario_name,
            "provider": config.provider,
            "model": config.model,
            "n_requests_submitted": 0,
            "plan_csv_path": str(plan_csv_path),
            "request_json_path": str(request_json_path),
        }

    batch = anthropic_client.messages.batches.create(requests=requests)

    batch_info = {
        "batch_id": batch.id,
        "scenario_name": scenario_name,
        "provider": config.provider,
        "model": config.model,
        "submitted_at_utc": now_iso(),
        "n_requests_submitted": len(requests),
        "plan_csv_path": str(plan_csv_path),
        "request_json_path": str(request_json_path),
        "processing_status": getattr(batch, "processing_status", None),
    }

    manifest_path = output_dir / f"{stem}__batch_manifest.json"
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(batch_info, f, indent=2)

    print("Submitted Anthropic batch:")
    print(json.dumps(batch_info, indent=2))

    return batch_info


def check_anthropic_batch(batch_id: str):
    batch = anthropic_client.messages.batches.retrieve(batch_id)
    print(batch)
    return batch


def check_anthropic_batches(batch_df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for _, row in batch_df.dropna(subset=["batch_id"]).iterrows():
        batch = anthropic_client.messages.batches.retrieve(row["batch_id"])
        request_counts = getattr(batch, "request_counts", None)

        rows.append({
            "batch_id": batch.id,
            "scenario_name": row["scenario_name"],
            "model": row["model"],
            "processing_status": getattr(batch, "processing_status", None),
            "request_counts": str(request_counts),
            "created_at": str(getattr(batch, "created_at", None)),
            "ended_at": str(getattr(batch, "ended_at", None)),
            "expires_at": str(getattr(batch, "expires_at", None)),
            "plan_csv_path": row["plan_csv_path"],
        })

    return pd.DataFrame(rows)


def extract_text_from_anthropic_message(message) -> str:
    texts = []

    for block in getattr(message, "content", []) or []:
        if getattr(block, "type", None) == "text":
            texts.append(getattr(block, "text", ""))

    return "\n".join(t for t in texts if t).strip()


def anthropic_usage_to_dict(message) -> dict:
    usage = getattr(message, "usage", None)
    if usage is None:
        return {}

    try:
        return usage.model_dump()
    except Exception:
        try:
            return dict(usage)
        except Exception:
            return {}


def parse_anthropic_batch_results_to_standard_jsonl(
    batch_id: str,
    plan_csv_path: str | Path,
    scenario_name: str,
    config: ProviderModelConfig,
    output_dir: Path = AI_DATA_DIR,
) -> Path:
    plan_df = pd.read_csv(plan_csv_path)
    plan_by_custom_id = {
        row["anthropic_custom_id"]: row.to_dict()
        for _, row in plan_df.iterrows()
    }

    standard_output_path = output_dir / f"{config.provider}__{config.model}__{scenario_name}.jsonl"
    raw_results_path = ANTHROPIC_BATCH_DIR / f"{batch_id}__results_raw.jsonl"

    n_success = 0
    n_error = 0

    result_stream = anthropic_client.messages.batches.results(batch_id)

    for entry in result_stream:
        try:
            entry_dict = entry.model_dump()
        except Exception:
            entry_dict = {"repr": repr(entry)}

        append_jsonl(raw_results_path, entry_dict)

        custom_id = getattr(entry, "custom_id", None)
        plan_row = plan_by_custom_id.get(custom_id, {})
        result = getattr(entry, "result", None)
        result_type = getattr(result, "type", None)

        if result_type == "succeeded":
            message = getattr(result, "message", None)
            text = extract_text_from_anthropic_message(message)
            usage = anthropic_usage_to_dict(message)

            record = {
                **plan_row,
                "created_at_utc": now_iso(),
                "status": "success" if text else "empty_text",
                "text": text if text else None,
                "provider_response_id": getattr(message, "id", None),
                "usage": usage,
                "raw_response": None,
                "error": None if text else "No text extracted from Anthropic message.",
                "batch_custom_id": custom_id,
                "batch_id": batch_id,
                "batch_output_file": str(raw_results_path),
            }

            n_success += int(bool(text))
            n_error += int(not bool(text))
        else:
            error_obj = getattr(result, "error", None)
            try:
                error_dump = error_obj.model_dump() if error_obj is not None else None
            except Exception:
                error_dump = repr(error_obj)

            record = {
                **plan_row,
                "created_at_utc": now_iso(),
                "status": "error",
                "text": None,
                "provider_response_id": None,
                "usage": None,
                "raw_response": None,
                "error": error_dump,
                "batch_custom_id": custom_id,
                "batch_id": batch_id,
                "batch_output_file": str(raw_results_path),
            }

            n_error += 1

        append_jsonl(standard_output_path, record)

    print(f"Raw Anthropic results: {raw_results_path}")
    print(f"Parsed standard output: {standard_output_path}")
    print(f"Success-ish text records: {n_success:,}")
    print(f"Errors/empty:             {n_error:,}")

    return standard_output_path

In [37]:
claude_cfg = MODEL_CONFIGS["claude_sonnet45"]

claude_main_batch = submit_anthropic_batch("neutral_main_t1", claude_cfg, aut_conditions_df)
claude_main_batch

Anthropic scenario: neutral_main_t1
Requests to submit: 250
Plan CSV: ai_data/aut_batches/anthropic/anthropic__claude-sonnet-4-5__aut__neutral_main_t1__20260429_124653__batch_plan.csv
Request JSON: ai_data/aut_batches/anthropic/anthropic__claude-sonnet-4-5__aut__neutral_main_t1__20260429_124653__batch_requests.json
Submitted Anthropic batch:
{
  "batch_id": "msgbatch_015ygCjDrG8UEGn5DxttFYfT",
  "scenario_name": "neutral_main_t1",
  "provider": "anthropic",
  "model": "claude-sonnet-4-5",
  "submitted_at_utc": "2026-04-29T16:46:53.971357+00:00",
  "n_requests_submitted": 250,
  "plan_csv_path": "ai_data/aut_batches/anthropic/anthropic__claude-sonnet-4-5__aut__neutral_main_t1__20260429_124653__batch_plan.csv",
  "request_json_path": "ai_data/aut_batches/anthropic/anthropic__claude-sonnet-4-5__aut__neutral_main_t1__20260429_124653__batch_requests.json",
  "processing_status": "in_progress"
}


{'batch_id': 'msgbatch_015ygCjDrG8UEGn5DxttFYfT',
 'scenario_name': 'neutral_main_t1',
 'provider': 'anthropic',
 'model': 'claude-sonnet-4-5',
 'submitted_at_utc': '2026-04-29T16:46:53.971357+00:00',
 'n_requests_submitted': 250,
 'plan_csv_path': 'ai_data/aut_batches/anthropic/anthropic__claude-sonnet-4-5__aut__neutral_main_t1__20260429_124653__batch_plan.csv',
 'request_json_path': 'ai_data/aut_batches/anthropic/anthropic__claude-sonnet-4-5__aut__neutral_main_t1__20260429_124653__batch_requests.json',
 'processing_status': 'in_progress'}

In [39]:
check_anthropic_batch(claude_main_batch["batch_id"])

MessageBatch(id='msgbatch_015ygCjDrG8UEGn5DxttFYfT', archived_at=None, cancel_initiated_at=None, created_at=datetime.datetime(2026, 4, 29, 16, 46, 53, 857366, tzinfo=datetime.timezone.utc), ended_at=datetime.datetime(2026, 4, 29, 16, 48, 46, 806223, tzinfo=TzInfo(UTC)), expires_at=datetime.datetime(2026, 4, 30, 16, 46, 53, 857366, tzinfo=datetime.timezone.utc), processing_status='ended', request_counts=MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=0, succeeded=250), results_url='https://api.anthropic.com/v1/messages/batches/msgbatch_015ygCjDrG8UEGn5DxttFYfT/results', type='message_batch')


MessageBatch(id='msgbatch_015ygCjDrG8UEGn5DxttFYfT', archived_at=None, cancel_initiated_at=None, created_at=datetime.datetime(2026, 4, 29, 16, 46, 53, 857366, tzinfo=datetime.timezone.utc), ended_at=datetime.datetime(2026, 4, 29, 16, 48, 46, 806223, tzinfo=TzInfo(UTC)), expires_at=datetime.datetime(2026, 4, 30, 16, 46, 53, 857366, tzinfo=datetime.timezone.utc), processing_status='ended', request_counts=MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=0, succeeded=250), results_url='https://api.anthropic.com/v1/messages/batches/msgbatch_015ygCjDrG8UEGn5DxttFYfT/results', type='message_batch')

In [40]:
batch = anthropic_client.messages.batches.retrieve(claude_main_batch["batch_id"])

if batch.processing_status != "ended":
    print("Claude main batch not ended yet.")
else:
    claude_main_standard_path = parse_anthropic_batch_results_to_standard_jsonl(
        batch_id=claude_main_batch["batch_id"],
        plan_csv_path=claude_main_batch["plan_csv_path"],
        scenario_name=claude_main_batch["scenario_name"],
        config=claude_cfg,
    )

    claude_main_df = load_generation_jsonl(claude_main_standard_path)
    print(claude_main_df.shape)
    print(claude_main_df["status"].value_counts(dropna=False))
    display(claude_main_df[["condition_id", "temperature", "run_idx", "status", "text"]].head(10))

Raw Anthropic results: ai_data/aut_batches/anthropic/msgbatch_015ygCjDrG8UEGn5DxttFYfT__results_raw.jsonl
Parsed standard output: ai_data/aut_generations/anthropic__claude-sonnet-4-5__neutral_main_t1.jsonl
Success-ish text records: 250
Errors/empty:             0
(250, 28)
status
success    250
Name: count, dtype: int64


,condition_id,temperature,run_idx,status,text
0,automobile_tire,1.0,0,success,A swing seat suspended by ropes from a tree br...
1,automobile_tire,1.0,1,success,A playground swing seat suspended by chains or...
2,automobile_tire,1.0,2,success,A playground swing seat suspended by chains or...
3,automobile_tire,1.0,3,success,A swing seat suspended from a tree branch by r...
4,automobile_tire,1.0,4,success,A raised garden bed by stacking and filling ti...
5,automobile_tire,1.0,5,success,A playground swing seat suspended by chains or...
6,automobile_tire,1.0,6,success,A playground swing seat suspended by chains or...
7,automobile_tire,1.0,7,success,A playground swing seat suspended by chains or...
8,automobile_tire,1.0,8,success,A playground swing seat suspended by chains or...
9,automobile_tire,1.0,9,success,A playground swing seat suspended by chains or...


In [41]:
claude_main_df = add_aut_validity_diagnostics(claude_main_df)

claude_main_df.groupby("condition_id").agg(
    n=("text", "size"),
    n_success=("status", lambda x: (x == "success").sum()),
    mean_words=("output_word_count", "mean"),
    median_words=("output_word_count", "median"),
    validity_rate=("valid_exactly_one_short_response_heuristic", "mean"),
)

,n,n_success,mean_words,median_words,validity_rate
condition_id,,,,,
automobile_tire,50,50,9.78,9.0,0.16
button,50,50,19.36,21.0,0.80
key,50,50,16.94,17.0,0.52
shoe,50,50,15.78,15.0,0.48
wooden_pencil,50,50,15.92,16.0,0.54


In [42]:
claude_temp_batch = submit_anthropic_batch("neutral_temperature_robustness", claude_cfg, aut_conditions_df)
claude_personality_batch = submit_anthropic_batch("personality_grid", claude_cfg, aut_conditions_df)

claude_batches = pd.DataFrame([claude_main_batch, claude_temp_batch, claude_personality_batch])
claude_manifest_path = ANTHROPIC_BATCH_DIR / "anthropic__claude-sonnet-4-5__aut_batch_manifest.csv"
claude_batches.to_csv(claude_manifest_path, index=False)

claude_batches

Anthropic scenario: neutral_temperature_robustness
Requests to submit: 100
Plan CSV: ai_data/aut_batches/anthropic/anthropic__claude-sonnet-4-5__aut__neutral_temperature_robustness__20260429_125140__batch_plan.csv
Request JSON: ai_data/aut_batches/anthropic/anthropic__claude-sonnet-4-5__aut__neutral_temperature_robustness__20260429_125140__batch_requests.json
Submitted Anthropic batch:
{
  "batch_id": "msgbatch_011Hrt45xTSjWgy13fyV3Qwr",
  "scenario_name": "neutral_temperature_robustness",
  "provider": "anthropic",
  "model": "claude-sonnet-4-5",
  "submitted_at_utc": "2026-04-29T16:51:41.284643+00:00",
  "n_requests_submitted": 100,
  "plan_csv_path": "ai_data/aut_batches/anthropic/anthropic__claude-sonnet-4-5__aut__neutral_temperature_robustness__20260429_125140__batch_plan.csv",
  "request_json_path": "ai_data/aut_batches/anthropic/anthropic__claude-sonnet-4-5__aut__neutral_temperature_robustness__20260429_125140__batch_requests.json",
  "processing_status": "in_progress"
}
Anthrop

,batch_id,scenario_name,provider,model,submitted_at_utc,n_requests_submitted,plan_csv_path,request_json_path,processing_status
0,msgbatch_015ygCjDrG8UEGn5DxttFYfT,neutral_main_t1,anthropic,claude-sonnet-4-5,2026-04-29T16:46:53.971357+00:00,250,ai_data/aut_batches/anthropic/anthropic__claud...,ai_data/aut_batches/anthropic/anthropic__claud...,in_progress
1,msgbatch_011Hrt45xTSjWgy13fyV3Qwr,neutral_temperature_robustness,anthropic,claude-sonnet-4-5,2026-04-29T16:51:41.284643+00:00,100,ai_data/aut_batches/anthropic/anthropic__claud...,ai_data/aut_batches/anthropic/anthropic__claud...,in_progress
2,msgbatch_01MaMHFkXBFgcctcEF2X9xmx,personality_grid,anthropic,claude-sonnet-4-5,2026-04-29T16:51:43.743469+00:00,4800,ai_data/aut_batches/anthropic/anthropic__claud...,ai_data/aut_batches/anthropic/anthropic__claud...,in_progress


In [45]:
check_anthropic_batches(claude_batches)

,batch_id,scenario_name,model,processing_status,request_counts,created_at,ended_at,expires_at,plan_csv_path
0,msgbatch_015ygCjDrG8UEGn5DxttFYfT,neutral_main_t1,claude-sonnet-4-5,ended,"MessageBatchRequestCounts(canceled=0, errored=...",2026-04-29 16:46:53.857366+00:00,2026-04-29 16:48:46.806223+00:00,2026-04-30 16:46:53.857366+00:00,ai_data/aut_batches/anthropic/anthropic__claud...
1,msgbatch_011Hrt45xTSjWgy13fyV3Qwr,neutral_temperature_robustness,claude-sonnet-4-5,ended,"MessageBatchRequestCounts(canceled=0, errored=...",2026-04-29 16:51:41.107817+00:00,2026-04-29 16:56:03.237389+00:00,2026-04-30 16:51:41.107817+00:00,ai_data/aut_batches/anthropic/anthropic__claud...
2,msgbatch_01MaMHFkXBFgcctcEF2X9xmx,personality_grid,claude-sonnet-4-5,ended,"MessageBatchRequestCounts(canceled=0, errored=...",2026-04-29 16:51:43.566367+00:00,2026-04-29 16:56:42.880491+00:00,2026-04-30 16:51:43.566367+00:00,ai_data/aut_batches/anthropic/anthropic__claud...


In [46]:
for batch_info in [claude_temp_batch, claude_personality_batch]:
    batch = anthropic_client.messages.batches.retrieve(batch_info["batch_id"])

    if batch.processing_status != "ended":
        print(f"Skipping {batch_info['scenario_name']}; status={batch.processing_status}")
        continue

    parse_anthropic_batch_results_to_standard_jsonl(
        batch_id=batch_info["batch_id"],
        plan_csv_path=batch_info["plan_csv_path"],
        scenario_name=batch_info["scenario_name"],
        config=claude_cfg,
    )

Raw Anthropic results: ai_data/aut_batches/anthropic/msgbatch_011Hrt45xTSjWgy13fyV3Qwr__results_raw.jsonl
Parsed standard output: ai_data/aut_generations/anthropic__claude-sonnet-4-5__neutral_temperature_robustness.jsonl
Success-ish text records: 100
Errors/empty:             0
Raw Anthropic results: ai_data/aut_batches/anthropic/msgbatch_01MaMHFkXBFgcctcEF2X9xmx__results_raw.jsonl
Parsed standard output: ai_data/aut_generations/anthropic__claude-sonnet-4-5__personality_grid.jsonl
Success-ish text records: 4,798
Errors/empty:             2


In [47]:
claude_personality_path = Path(
    "ai_data/aut_generations/anthropic__claude-sonnet-4-5__personality_grid.jsonl"
)

claude_personality_df = load_generation_jsonl(claude_personality_path)

print(claude_personality_df.shape)
print(claude_personality_df["status"].value_counts(dropna=False))

claude_bad = claude_personality_df.query("status != 'success'").copy()
claude_bad[
    [
        "condition_id",
        "object",
        "temperature",
        "persona_id",
        "run_idx",
        "status",
        "error",
    ]
]

(4800, 28)
status
success       4798
empty_text       2
Name: count, dtype: int64


,condition_id,object,temperature,persona_id,run_idx,status,error
600,automobile_tire,automobile tire,0.7,introverted__antagonistic__unconscientious__ne...,0,empty_text,No text extracted from Anthropic message.
1845,button,button,1.0,introverted__antagonistic__conscientious__neur...,5,empty_text,No text extracted from Anthropic message.


In [48]:
for _, row in claude_bad.iterrows():
    print("=" * 100)
    print("condition_id:", row["condition_id"])
    print("temperature:", row["temperature"])
    print("persona_id:", row["persona_id"])
    print("run_idx:", row["run_idx"])
    print("status:", row["status"])
    print("error:")
    print(json.dumps(row["error"], indent=2) if isinstance(row["error"], dict) else row["error"])

condition_id: automobile_tire
temperature: 0.7
persona_id: introverted__antagonistic__unconscientious__neurotic__open_to_experience
run_idx: 0
status: empty_text
error:
No text extracted from Anthropic message.
condition_id: button
temperature: 1.0
persona_id: introverted__antagonistic__conscientious__neurotic__open_to_experience
run_idx: 5
status: empty_text
error:
No text extracted from Anthropic message.


In [49]:
claude_all_df = load_all_generation_files(
    provider_name="anthropic",
    model_name="claude-sonnet-4-5",
    output_dir=AI_DATA_DIR,
)

claude_all_df = deduplicate_generation_records(claude_all_df)
claude_all_df = add_aut_validity_diagnostics(claude_all_df)

print("Shape:", claude_all_df.shape)
print(claude_all_df["scenario_name"].value_counts(dropna=False))
print(claude_all_df["status"].value_counts(dropna=False))

claude_all_df.groupby(["scenario_name", "temperature", "status"], dropna=False).size()

Shape: (5150, 35)
scenario_name
personality_grid                  4800
neutral_main_t1                    250
neutral_temperature_robustness     100
Name: count, dtype: int64
status
success       5148
empty_text       2
Name: count, dtype: int64


scenario_name                   temperature  status    
neutral_main_t1                 1.0          success        250
neutral_temperature_robustness  0.3          success         50
                                0.7          success         50
personality_grid                0.3          success       1600
                                0.7          empty_text       1
                                             success       1599
                                1.0          empty_text       1
                                             success       1599
dtype: int64

In [50]:
claude_personality_coverage = (
    claude_all_df
    .query("scenario_name == 'personality_grid'")
    .groupby(["condition_id", "temperature", "persona_id"], dropna=False)
    .agg(
        n_records=("request_key", "size"),
        n_success=("status", lambda x: (x == "success").sum()),
    )
    .reset_index()
)

incomplete_claude_personality_cells = claude_personality_coverage.query("n_success < 10")

print(f"Incomplete Claude personality cells: {len(incomplete_claude_personality_cells)}")
incomplete_claude_personality_cells

Incomplete Claude personality cells: 2


,condition_id,temperature,persona_id,n_records,n_success
63,automobile_tire,0.7,introverted__antagonistic__unconscientious__ne...,10,9
187,button,1.0,introverted__antagonistic__conscientious__neur...,10,9


In [51]:
claude_cfg = MODEL_CONFIGS["claude_sonnet45"]

claude_personality_backfill = submit_anthropic_batch(
    scenario_name="personality_grid",
    config=claude_cfg,
    conditions_df=aut_conditions_df,
    exclude_existing_successes=True,
)

claude_personality_backfill

Anthropic scenario: personality_grid
Requests to submit: 2
Plan CSV: ai_data/aut_batches/anthropic/anthropic__claude-sonnet-4-5__aut__personality_grid__20260429_131057__batch_plan.csv
Request JSON: ai_data/aut_batches/anthropic/anthropic__claude-sonnet-4-5__aut__personality_grid__20260429_131057__batch_requests.json
Submitted Anthropic batch:
{
  "batch_id": "msgbatch_01H85uGWsR7qqoPKqcuqjHHW",
  "scenario_name": "personality_grid",
  "provider": "anthropic",
  "model": "claude-sonnet-4-5",
  "submitted_at_utc": "2026-04-29T17:10:58.096289+00:00",
  "n_requests_submitted": 2,
  "plan_csv_path": "ai_data/aut_batches/anthropic/anthropic__claude-sonnet-4-5__aut__personality_grid__20260429_131057__batch_plan.csv",
  "request_json_path": "ai_data/aut_batches/anthropic/anthropic__claude-sonnet-4-5__aut__personality_grid__20260429_131057__batch_requests.json",
  "processing_status": "in_progress"
}


{'batch_id': 'msgbatch_01H85uGWsR7qqoPKqcuqjHHW',
 'scenario_name': 'personality_grid',
 'provider': 'anthropic',
 'model': 'claude-sonnet-4-5',
 'submitted_at_utc': '2026-04-29T17:10:58.096289+00:00',
 'n_requests_submitted': 2,
 'plan_csv_path': 'ai_data/aut_batches/anthropic/anthropic__claude-sonnet-4-5__aut__personality_grid__20260429_131057__batch_plan.csv',
 'request_json_path': 'ai_data/aut_batches/anthropic/anthropic__claude-sonnet-4-5__aut__personality_grid__20260429_131057__batch_requests.json',
 'processing_status': 'in_progress'}

In [61]:
check_anthropic_batch(claude_personality_backfill["batch_id"])

MessageBatch(id='msgbatch_01H85uGWsR7qqoPKqcuqjHHW', archived_at=None, cancel_initiated_at=None, created_at=datetime.datetime(2026, 4, 29, 17, 10, 57, 977707, tzinfo=datetime.timezone.utc), ended_at=datetime.datetime(2026, 4, 29, 17, 45, 12, 991836, tzinfo=TzInfo(UTC)), expires_at=datetime.datetime(2026, 4, 30, 17, 10, 57, 977707, tzinfo=datetime.timezone.utc), processing_status='ended', request_counts=MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=0, succeeded=2), results_url='https://api.anthropic.com/v1/messages/batches/msgbatch_01H85uGWsR7qqoPKqcuqjHHW/results', type='message_batch')


MessageBatch(id='msgbatch_01H85uGWsR7qqoPKqcuqjHHW', archived_at=None, cancel_initiated_at=None, created_at=datetime.datetime(2026, 4, 29, 17, 10, 57, 977707, tzinfo=datetime.timezone.utc), ended_at=datetime.datetime(2026, 4, 29, 17, 45, 12, 991836, tzinfo=TzInfo(UTC)), expires_at=datetime.datetime(2026, 4, 30, 17, 10, 57, 977707, tzinfo=datetime.timezone.utc), processing_status='ended', request_counts=MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=0, succeeded=2), results_url='https://api.anthropic.com/v1/messages/batches/msgbatch_01H85uGWsR7qqoPKqcuqjHHW/results', type='message_batch')

In [62]:
batch = anthropic_client.messages.batches.retrieve(
    claude_personality_backfill["batch_id"]
)

if batch.processing_status != "ended":
    print(f"Backfill not ended yet: {batch.processing_status}")
else:
    claude_backfill_standard_path = parse_anthropic_batch_results_to_standard_jsonl(
        batch_id=claude_personality_backfill["batch_id"],
        plan_csv_path=claude_personality_backfill["plan_csv_path"],
        scenario_name=claude_personality_backfill["scenario_name"],
        config=claude_cfg,
    )

    claude_backfill_df = load_generation_jsonl(claude_backfill_standard_path)
    print(claude_backfill_df.shape)
    print(claude_backfill_df["status"].value_counts(dropna=False))

Raw Anthropic results: ai_data/aut_batches/anthropic/msgbatch_01H85uGWsR7qqoPKqcuqjHHW__results_raw.jsonl
Parsed standard output: ai_data/aut_generations/anthropic__claude-sonnet-4-5__personality_grid.jsonl
Success-ish text records: 2
Errors/empty:             0
(4802, 28)
status
success       4800
empty_text       2
Name: count, dtype: int64


In [63]:
claude_all_df = load_all_generation_files(
    provider_name="anthropic",
    model_name="claude-sonnet-4-5",
    output_dir=AI_DATA_DIR,
)

claude_all_df = deduplicate_generation_records(claude_all_df)
claude_all_df = add_aut_validity_diagnostics(claude_all_df)

print("Final Claude AUT shape:", claude_all_df.shape)
print(claude_all_df["scenario_name"].value_counts(dropna=False))
print(claude_all_df["status"].value_counts(dropna=False))

print(
    claude_all_df
    .query("status == 'success'")
    .groupby(["scenario_name", "temperature"], dropna=False)
    .size()
)

save_provider_outputs("anthropic", "claude-sonnet-4-5", claude_all_df)

Final Claude AUT shape: (5150, 35)
scenario_name
personality_grid                  4800
neutral_main_t1                    250
neutral_temperature_robustness     100
Name: count, dtype: int64
status
success    5150
Name: count, dtype: int64
scenario_name                   temperature
neutral_main_t1                 1.0             250
neutral_temperature_robustness  0.3              50
                                0.7              50
personality_grid                0.3            1600
                                0.7            1600
                                1.0            1600
dtype: int64
Saved:
ai_data/processed/anthropic__claude-sonnet-4-5__aut_generations_all_records_with_errors.pkl
ai_data/processed/anthropic__claude-sonnet-4-5__aut_generations_all_records_with_errors.csv
ai_data/processed/anthropic__claude-sonnet-4-5__aut_generations_success_only.pkl
ai_data/processed/anthropic__claude-sonnet-4-5__aut_generations_success_only.csv


{'audit_pkl': PosixPath('ai_data/processed/anthropic__claude-sonnet-4-5__aut_generations_all_records_with_errors.pkl'),
 'audit_csv': PosixPath('ai_data/processed/anthropic__claude-sonnet-4-5__aut_generations_all_records_with_errors.csv'),
 'success_pkl': PosixPath('ai_data/processed/anthropic__claude-sonnet-4-5__aut_generations_success_only.pkl'),
 'success_csv': PosixPath('ai_data/processed/anthropic__claude-sonnet-4-5__aut_generations_success_only.csv')}

# Gemini

In [64]:
def make_gemini_custom_id(request_key: str) -> str:
    return "r_" + request_key[:48]


def build_gemini_generation_plan(
    scenario_name: str,
    config: ProviderModelConfig,
    conditions_df: pd.DataFrame,
) -> pd.DataFrame:
    scenarios = build_scenarios_for_provider(config)
    scenario = scenarios[scenario_name]

    plan_df = build_aut_generation_plan(
        scenario=scenario,
        conditions_df=conditions_df,
        provider_name=config.provider,
        model_name=config.model,
    ).copy()

    plan_df["gemini_custom_id"] = plan_df["request_key"].map(make_gemini_custom_id)

    if plan_df["gemini_custom_id"].duplicated().any():
        raise ValueError("Duplicate gemini_custom_id found. Increase hash length.")

    return plan_df


def make_gemini_batch_jsonl_file(
    scenario_name: str,
    config: ProviderModelConfig,
    conditions_df: pd.DataFrame,
    output_dir: Path = GEMINI_BATCH_DIR,
    exclude_existing_successes: bool = True,
    thinking_budget: int = 0,
) -> tuple[Path, Path, pd.DataFrame]:
    standard_output_path = AI_DATA_DIR / f"{config.provider}__{config.model}__{scenario_name}.jsonl"

    plan_df = build_gemini_generation_plan(
        scenario_name=scenario_name,
        config=config,
        conditions_df=conditions_df,
    )

    if exclude_existing_successes and standard_output_path.exists():
        done_keys = successful_request_keys(standard_output_path)
        plan_df = plan_df[~plan_df["request_key"].isin(done_keys)].copy()

    timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    stem = f"{config.provider}__{config.model}__{TASK_FAMILY}__{scenario_name}__{timestamp}"

    plan_csv_path = output_dir / f"{stem}__batch_plan.csv"
    batch_jsonl_path = output_dir / f"{stem}__batch_input.jsonl"

    plan_df.to_csv(plan_csv_path, index=False)

    with open(batch_jsonl_path, "w", encoding="utf-8") as f:
        for _, row in plan_df.iterrows():
            request_obj = {
                "key": row["gemini_custom_id"],
                "request": {
                    "contents": [
                        {
                            "role": "user",
                            "parts": [{"text": row["user_prompt"]}],
                        }
                    ],
                    "systemInstruction": {
                        "parts": [{"text": row["system_instructions"]}]
                    },
                    "generationConfig": {
                        "temperature": float(row["temperature"]),
                        "maxOutputTokens": int(row["max_output_tokens"]),
                        "responseModalities": ["TEXT"],
                        "thinkingConfig": {
                            "thinkingBudget": int(thinking_budget)
                        },
                    },
                },
            }

            f.write(json.dumps(request_obj, ensure_ascii=False) + "\n")

    print(f"Gemini scenario: {scenario_name}")
    print(f"Requests written: {len(plan_df):,}")
    print(f"Model: {config.model}")
    print(f"maxOutputTokens: {config.max_output_tokens}")
    print(f"thinkingBudget: {thinking_budget}")
    print(f"Plan CSV:     {plan_csv_path}")
    print(f"Batch JSONL:  {batch_jsonl_path}")

    return batch_jsonl_path, plan_csv_path, plan_df


def submit_gemini_batch_from_jsonl(
    scenario_name: str,
    config: ProviderModelConfig,
    conditions_df: pd.DataFrame,
    output_dir: Path = GEMINI_BATCH_DIR,
    exclude_existing_successes: bool = True,
) -> dict:
    batch_jsonl_path, plan_csv_path, plan_df = make_gemini_batch_jsonl_file(
        scenario_name=scenario_name,
        config=config,
        conditions_df=conditions_df,
        output_dir=output_dir,
        exclude_existing_successes=exclude_existing_successes,
        thinking_budget=0,
    )

    if len(plan_df) == 0:
        return {
            "batch_name": None,
            "scenario_name": scenario_name,
            "provider": config.provider,
            "model": config.model,
            "submitted_at_utc": now_iso(),
            "n_requests_submitted": 0,
            "plan_csv_path": str(plan_csv_path),
            "batch_jsonl_path": str(batch_jsonl_path),
            "uploaded_file_name": None,
        }

    uploaded_file = gemini_client.files.upload(
        file=str(batch_jsonl_path),
        config={"mime_type": "application/jsonl"},
    )

    print(f"Uploaded file: {uploaded_file.name}")

    batch_job = gemini_client.batches.create(
        model=config.model,
        src=uploaded_file.name,
        config={"display_name": batch_jsonl_path.stem},
    )

    batch_info = {
        "batch_name": batch_job.name,
        "scenario_name": scenario_name,
        "provider": config.provider,
        "model": config.model,
        "submitted_at_utc": now_iso(),
        "n_requests_submitted": len(plan_df),
        "plan_csv_path": str(plan_csv_path),
        "batch_jsonl_path": str(batch_jsonl_path),
        "uploaded_file_name": uploaded_file.name,
        "raw_state": str(getattr(batch_job, "state", None)),
    }

    manifest_path = output_dir / f"{batch_jsonl_path.stem}__batch_manifest.json"
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(batch_info, f, indent=2)

    print("Submitted Gemini batch:")
    print(json.dumps(batch_info, indent=2))

    return batch_info


def check_gemini_batch(batch_name: str):
    batch = gemini_client.batches.get(name=batch_name)
    print(batch)
    return batch


def download_gemini_batch_output_file(batch_name: str, output_dir: Path = GEMINI_BATCH_DIR) -> Path:
    job = gemini_client.batches.get(name=batch_name)

    print("Batch state:", getattr(job, "state", None))
    print("Batch error:", getattr(job, "error", None))
    print("Batch dest:", getattr(job, "dest", None))

    dest = getattr(job, "dest", None)
    file_name = getattr(dest, "file_name", None)

    if not file_name:
        raise ValueError(f"No dest.file_name found on batch job: {batch_name}")

    safe_name = batch_name.replace("/", "_")
    output_path = output_dir / f"{safe_name}__output.jsonl"

    downloaded_bytes = gemini_client.files.download(file=file_name)

    if not isinstance(downloaded_bytes, (bytes, bytearray)):
        raise TypeError(f"Expected bytes from files.download, got {type(downloaded_bytes)}")

    output_path.write_bytes(downloaded_bytes)

    print(f"Downloaded Gemini batch output to: {output_path}")
    print(f"File size: {output_path.stat().st_size:,} bytes")

    return output_path


def extract_text_from_gemini_response_dict(response: dict) -> str:
    if not isinstance(response, dict):
        return ""

    if response.get("text"):
        return str(response["text"]).strip()

    texts = []

    for cand in response.get("candidates", []) or []:
        content = cand.get("content", {}) or {}
        for part in content.get("parts", []) or []:
            if isinstance(part, dict) and part.get("text"):
                texts.append(part["text"])

    return "\n".join(texts).strip()


def extract_gemini_usage_dict(response: dict) -> dict:
    if not isinstance(response, dict):
        return {}

    return (
        response.get("usageMetadata")
        or response.get("usage_metadata")
        or response.get("usage")
        or {}
    )


def parse_gemini_downloaded_batch_jsonl_to_standard_jsonl(
    downloaded_output_path: Path,
    plan_csv_path: str | Path,
    scenario_name: str,
    config: ProviderModelConfig,
    output_dir: Path = AI_DATA_DIR,
) -> Path:
    plan_df = pd.read_csv(plan_csv_path)
    plan_by_custom_id = {
        row["gemini_custom_id"]: row.to_dict()
        for _, row in plan_df.iterrows()
    }

    raw_records = read_jsonl(downloaded_output_path)

    standard_output_path = output_dir / f"{config.provider}__{config.model}__{scenario_name}.jsonl"

    n_success = 0
    n_error = 0

    for rec in raw_records:
        custom_id = rec.get("key") or rec.get("custom_id") or rec.get("customId")
        plan_row = plan_by_custom_id.get(custom_id, {})

        response = rec.get("response") or rec.get("result") or {}
        error = rec.get("error")

        if isinstance(response, dict) and "body" in response:
            response = response.get("body") or {}

        text = extract_text_from_gemini_response_dict(response)
        usage = extract_gemini_usage_dict(response)

        if text:
            status = "success"
            n_success += 1
            error_out = None
        else:
            status = "error" if error else "empty_text"
            n_error += 1
            error_out = error or "No text extracted from Gemini response."

        record = {
            **plan_row,
            "created_at_utc": now_iso(),
            "status": status,
            "text": text if text else None,
            "provider_response_id": response.get("responseId") if isinstance(response, dict) else None,
            "usage": usage,
            "raw_response": None,
            "error": error_out,
            "batch_custom_id": custom_id,
            "batch_name": None,
            "batch_output_file": str(downloaded_output_path),
        }

        append_jsonl(standard_output_path, record)

    print(f"Parsed Gemini batch output to: {standard_output_path}")
    print(f"Success-ish text records: {n_success:,}")
    print(f"Errors/empty:             {n_error:,}")

    return standard_output_path

In [65]:
gemini_cfg = MODEL_CONFIGS["gemini_flash25"]

test_condition = aut_conditions_df.iloc[0]

test_response = gemini_client.models.generate_content(
    model=gemini_cfg.model,
    contents=[
        types.Content(
            role="user",
            parts=[types.Part(text=build_aut_user_prompt(test_condition["object"], test_condition["common_use"]))]
        )
    ],
    config=types.GenerateContentConfig(
        system_instruction=AUT_NEUTRAL_SYSTEM_INSTRUCTIONS,
        temperature=1.3,
        max_output_tokens=120,
        thinking_config=types.ThinkingConfig(thinking_budget=0),
    ),
)

print(test_response.text)

A weighted component in a homemade wind chime.


In [66]:
gemini_main_batch = submit_gemini_batch_from_jsonl(
    scenario_name="neutral_main_t1",
    config=gemini_cfg,
    conditions_df=aut_conditions_df,
)

gemini_main_batch

Gemini scenario: neutral_main_t1
Requests written: 250
Model: gemini-2.5-flash
maxOutputTokens: 120
thinkingBudget: 0
Plan CSV:     ai_data/aut_batches/gemini/gemini__gemini-2.5-flash__aut__neutral_main_t1__20260429_140033__batch_plan.csv
Batch JSONL:  ai_data/aut_batches/gemini/gemini__gemini-2.5-flash__aut__neutral_main_t1__20260429_140033__batch_input.jsonl
Uploaded file: files/bv67wu5ra97u
Submitted Gemini batch:
{
  "batch_name": "batches/3yickcajau3h3sx2jioz3uybnz6w17d1u3d7",
  "scenario_name": "neutral_main_t1",
  "provider": "gemini",
  "model": "gemini-2.5-flash",
  "submitted_at_utc": "2026-04-29T18:00:36.180675+00:00",
  "n_requests_submitted": 250,
  "plan_csv_path": "ai_data/aut_batches/gemini/gemini__gemini-2.5-flash__aut__neutral_main_t1__20260429_140033__batch_plan.csv",
  "batch_jsonl_path": "ai_data/aut_batches/gemini/gemini__gemini-2.5-flash__aut__neutral_main_t1__20260429_140033__batch_input.jsonl",
  "uploaded_file_name": "files/bv67wu5ra97u",
  "raw_state": "JobSt

{'batch_name': 'batches/3yickcajau3h3sx2jioz3uybnz6w17d1u3d7',
 'scenario_name': 'neutral_main_t1',
 'provider': 'gemini',
 'model': 'gemini-2.5-flash',
 'submitted_at_utc': '2026-04-29T18:00:36.180675+00:00',
 'n_requests_submitted': 250,
 'plan_csv_path': 'ai_data/aut_batches/gemini/gemini__gemini-2.5-flash__aut__neutral_main_t1__20260429_140033__batch_plan.csv',
 'batch_jsonl_path': 'ai_data/aut_batches/gemini/gemini__gemini-2.5-flash__aut__neutral_main_t1__20260429_140033__batch_input.jsonl',
 'uploaded_file_name': 'files/bv67wu5ra97u',
 'raw_state': 'JobState.JOB_STATE_PENDING'}

In [69]:
check_gemini_batch(gemini_main_batch["batch_name"])

name='batches/3yickcajau3h3sx2jioz3uybnz6w17d1u3d7' display_name='gemini__gemini-2.5-flash__aut__neutral_main_t1__20260429_140033__batch_input' state=<JobState.JOB_STATE_SUCCEEDED: 'JOB_STATE_SUCCEEDED'> error=None create_time=datetime.datetime(2026, 4, 29, 18, 0, 35, 573154, tzinfo=TzInfo(UTC)) start_time=None end_time=datetime.datetime(2026, 4, 29, 18, 2, 45, 219974, tzinfo=TzInfo(UTC)) update_time=datetime.datetime(2026, 4, 29, 18, 2, 45, 219974, tzinfo=TzInfo(UTC)) model='models/gemini-2.5-flash' src=None dest=BatchJobDestination(
  file_name='files/batch-3yickcajau3h3sx2jioz3uybnz6w17d1u3d7'
)


BatchJob(
  create_time=datetime.datetime(2026, 4, 29, 18, 0, 35, 573154, tzinfo=TzInfo(UTC)),
  dest=BatchJobDestination(
    file_name='files/batch-3yickcajau3h3sx2jioz3uybnz6w17d1u3d7'
  ),
  display_name='gemini__gemini-2.5-flash__aut__neutral_main_t1__20260429_140033__batch_input',
  end_time=datetime.datetime(2026, 4, 29, 18, 2, 45, 219974, tzinfo=TzInfo(UTC)),
  model='models/gemini-2.5-flash',
  name='batches/3yickcajau3h3sx2jioz3uybnz6w17d1u3d7',
  state=<JobState.JOB_STATE_SUCCEEDED: 'JOB_STATE_SUCCEEDED'>,
  update_time=datetime.datetime(2026, 4, 29, 18, 2, 45, 219974, tzinfo=TzInfo(UTC))
)

In [70]:
gemini_main_output_path = download_gemini_batch_output_file(
    gemini_main_batch["batch_name"],
    output_dir=GEMINI_BATCH_DIR,
)

gemini_main_standard_path = parse_gemini_downloaded_batch_jsonl_to_standard_jsonl(
    downloaded_output_path=gemini_main_output_path,
    plan_csv_path=gemini_main_batch["plan_csv_path"],
    scenario_name=gemini_main_batch["scenario_name"],
    config=gemini_cfg,
)

gemini_main_df = load_generation_jsonl(gemini_main_standard_path)

print(gemini_main_df.shape)
print(gemini_main_df["status"].value_counts(dropna=False))
display(gemini_main_df[["condition_id", "temperature", "run_idx", "status", "text"]].head(10))

Batch state: JobState.JOB_STATE_SUCCEEDED
Batch error: None
Batch dest: format=None gcs_uri=None bigquery_uri=None file_name='files/batch-3yickcajau3h3sx2jioz3uybnz6w17d1u3d7' inlined_responses=None inlined_embed_content_responses=None
Downloaded Gemini batch output to: ai_data/aut_batches/gemini/batches_3yickcajau3h3sx2jioz3uybnz6w17d1u3d7__output.jsonl
File size: 115,005 bytes
Parsed Gemini batch output to: ai_data/aut_generations/gemini__gemini-2.5-flash__neutral_main_t1.jsonl
Success-ish text records: 250
Errors/empty:             0
(250, 28)
status
success    250
Name: count, dtype: int64


,condition_id,temperature,run_idx,status,text
0,automobile_tire,1.0,0,success,A child's swing seat.
1,automobile_tire,1.0,1,success,A tire could be sliced into rings and used as ...
2,automobile_tire,1.0,2,success,A durable and flexible base for crafting an ou...
3,automobile_tire,1.0,3,success,A heavy-duty outdoor swing for a playground.
4,automobile_tire,1.0,4,success,A stack of tires can be used to create an impa...
5,automobile_tire,1.0,5,success,A durable base for a homemade composting bin.
6,automobile_tire,1.0,6,success,"Cut into strips and woven into a durable, all-..."
7,automobile_tire,1.0,7,success,A tire can be sliced into rings to create dura...
8,automobile_tire,1.0,8,success,A tire swing for a playground.
9,automobile_tire,1.0,9,success,As a weighted base for a temporary outdoor sign.


In [71]:
gemini_main_df = add_aut_validity_diagnostics(gemini_main_df)

gemini_main_df.groupby("condition_id").agg(
    n=("text", "size"),
    n_success=("status", lambda x: (x == "success").sum()),
    mean_words=("output_word_count", "mean"),
    median_words=("output_word_count", "median"),
    validity_rate=("valid_exactly_one_short_response_heuristic", "mean"),
)

,n,n_success,mean_words,median_words,validity_rate
condition_id,,,,,
automobile_tire,50,50,12.92,11.5,0.86
button,50,50,15.08,14.0,0.88
key,50,50,9.78,9.0,0.98
shoe,50,50,14.54,15.0,0.70
wooden_pencil,50,50,10.86,10.0,0.92


In [72]:
gemini_temp_batch = submit_gemini_batch_from_jsonl(
    scenario_name="neutral_temperature_robustness",
    config=gemini_cfg,
    conditions_df=aut_conditions_df,
)

gemini_personality_batch = submit_gemini_batch_from_jsonl(
    scenario_name="personality_grid",
    config=gemini_cfg,
    conditions_df=aut_conditions_df,
)

gemini_batches = pd.DataFrame([gemini_main_batch, gemini_temp_batch, gemini_personality_batch])
gemini_manifest_path = GEMINI_BATCH_DIR / "gemini__gemini-2.5-flash__aut_batch_manifest.csv"
gemini_batches.to_csv(gemini_manifest_path, index=False)

gemini_batches

Gemini scenario: neutral_temperature_robustness
Requests written: 100
Model: gemini-2.5-flash
maxOutputTokens: 120
thinkingBudget: 0
Plan CSV:     ai_data/aut_batches/gemini/gemini__gemini-2.5-flash__aut__neutral_temperature_robustness__20260429_141020__batch_plan.csv
Batch JSONL:  ai_data/aut_batches/gemini/gemini__gemini-2.5-flash__aut__neutral_temperature_robustness__20260429_141020__batch_input.jsonl
Uploaded file: files/ws5ju92vf2uc
Submitted Gemini batch:
{
  "batch_name": "batches/ijbiafndno77hnd444auyonzi9vybq41m54c",
  "scenario_name": "neutral_temperature_robustness",
  "provider": "gemini",
  "model": "gemini-2.5-flash",
  "submitted_at_utc": "2026-04-29T18:10:22.873526+00:00",
  "n_requests_submitted": 100,
  "plan_csv_path": "ai_data/aut_batches/gemini/gemini__gemini-2.5-flash__aut__neutral_temperature_robustness__20260429_141020__batch_plan.csv",
  "batch_jsonl_path": "ai_data/aut_batches/gemini/gemini__gemini-2.5-flash__aut__neutral_temperature_robustness__20260429_14102

,batch_name,scenario_name,provider,model,submitted_at_utc,n_requests_submitted,plan_csv_path,batch_jsonl_path,uploaded_file_name,raw_state
0,batches/3yickcajau3h3sx2jioz3uybnz6w17d1u3d7,neutral_main_t1,gemini,gemini-2.5-flash,2026-04-29T18:00:36.180675+00:00,250,ai_data/aut_batches/gemini/gemini__gemini-2.5-...,ai_data/aut_batches/gemini/gemini__gemini-2.5-...,files/bv67wu5ra97u,JobState.JOB_STATE_PENDING
1,batches/ijbiafndno77hnd444auyonzi9vybq41m54c,neutral_temperature_robustness,gemini,gemini-2.5-flash,2026-04-29T18:10:22.873526+00:00,100,ai_data/aut_batches/gemini/gemini__gemini-2.5-...,ai_data/aut_batches/gemini/gemini__gemini-2.5-...,files/ws5ju92vf2uc,JobState.JOB_STATE_PENDING
2,batches/aoqh19h2drajz29klbx0ufedt49by60abzxv,personality_grid,gemini,gemini-2.5-flash,2026-04-29T18:10:25.941718+00:00,4800,ai_data/aut_batches/gemini/gemini__gemini-2.5-...,ai_data/aut_batches/gemini/gemini__gemini-2.5-...,files/wb8d0mg1vadq,JobState.JOB_STATE_PENDING


In [74]:
for _, row in gemini_batches.dropna(subset=["batch_name"]).iterrows():
    print("=" * 100)
    print(row["scenario_name"])
    check_gemini_batch(row["batch_name"])

neutral_main_t1
name='batches/3yickcajau3h3sx2jioz3uybnz6w17d1u3d7' display_name='gemini__gemini-2.5-flash__aut__neutral_main_t1__20260429_140033__batch_input' state=<JobState.JOB_STATE_SUCCEEDED: 'JOB_STATE_SUCCEEDED'> error=None create_time=datetime.datetime(2026, 4, 29, 18, 0, 35, 573154, tzinfo=TzInfo(UTC)) start_time=None end_time=datetime.datetime(2026, 4, 29, 18, 2, 45, 219974, tzinfo=TzInfo(UTC)) update_time=datetime.datetime(2026, 4, 29, 18, 2, 45, 219974, tzinfo=TzInfo(UTC)) model='models/gemini-2.5-flash' src=None dest=BatchJobDestination(
  file_name='files/batch-3yickcajau3h3sx2jioz3uybnz6w17d1u3d7'
)
neutral_temperature_robustness
name='batches/ijbiafndno77hnd444auyonzi9vybq41m54c' display_name='gemini__gemini-2.5-flash__aut__neutral_temperature_robustness__20260429_141020__batch_input' state=<JobState.JOB_STATE_SUCCEEDED: 'JOB_STATE_SUCCEEDED'> error=None create_time=datetime.datetime(2026, 4, 29, 18, 10, 22, 287214, tzinfo=TzInfo(UTC)) start_time=None end_time=datetime.

In [75]:
for batch_info in [gemini_temp_batch, gemini_personality_batch]:
    job = gemini_client.batches.get(name=batch_info["batch_name"])

    if str(getattr(job, "state", "")) != "JobState.JOB_STATE_SUCCEEDED":
        print(f"Skipping {batch_info['scenario_name']}; state={getattr(job, 'state', None)}")
        continue

    output_path = download_gemini_batch_output_file(
        batch_info["batch_name"],
        output_dir=GEMINI_BATCH_DIR,
    )

    parse_gemini_downloaded_batch_jsonl_to_standard_jsonl(
        downloaded_output_path=output_path,
        plan_csv_path=batch_info["plan_csv_path"],
        scenario_name=batch_info["scenario_name"],
        config=gemini_cfg,
    )

Batch state: JobState.JOB_STATE_SUCCEEDED
Batch error: None
Batch dest: format=None gcs_uri=None bigquery_uri=None file_name='files/batch-ijbiafndno77hnd444auyonzi9vybq41m54c' inlined_responses=None inlined_embed_content_responses=None
Downloaded Gemini batch output to: ai_data/aut_batches/gemini/batches_ijbiafndno77hnd444auyonzi9vybq41m54c__output.jsonl
File size: 46,063 bytes
Parsed Gemini batch output to: ai_data/aut_generations/gemini__gemini-2.5-flash__neutral_temperature_robustness.jsonl
Success-ish text records: 100
Errors/empty:             0
Batch state: JobState.JOB_STATE_SUCCEEDED
Batch error: None
Batch dest: format=None gcs_uri=None bigquery_uri=None file_name='files/batch-aoqh19h2drajz29klbx0ufedt49by60abzxv' inlined_responses=None inlined_embed_content_responses=None
Downloaded Gemini batch output to: ai_data/aut_batches/gemini/batches_aoqh19h2drajz29klbx0ufedt49by60abzxv__output.jsonl
File size: 2,331,122 bytes
Parsed Gemini batch output to: ai_data/aut_generations/gemi

In [76]:
gemini_all_df = load_all_generation_files("gemini", "gemini-2.5-flash", AI_DATA_DIR)
gemini_all_df = deduplicate_generation_records(gemini_all_df)
gemini_all_df = add_aut_validity_diagnostics(gemini_all_df)

print(gemini_all_df["scenario_name"].value_counts(dropna=False))
print(gemini_all_df["status"].value_counts(dropna=False))

save_provider_outputs("gemini", "gemini-2.5-flash", gemini_all_df)

scenario_name
personality_grid                  4800
neutral_main_t1                    250
neutral_temperature_robustness     100
Name: count, dtype: int64
status
success    5150
Name: count, dtype: int64
Saved:
ai_data/processed/gemini__gemini-2.5-flash__aut_generations_all_records_with_errors.pkl
ai_data/processed/gemini__gemini-2.5-flash__aut_generations_all_records_with_errors.csv
ai_data/processed/gemini__gemini-2.5-flash__aut_generations_success_only.pkl
ai_data/processed/gemini__gemini-2.5-flash__aut_generations_success_only.csv


{'audit_pkl': PosixPath('ai_data/processed/gemini__gemini-2.5-flash__aut_generations_all_records_with_errors.pkl'),
 'audit_csv': PosixPath('ai_data/processed/gemini__gemini-2.5-flash__aut_generations_all_records_with_errors.csv'),
 'success_pkl': PosixPath('ai_data/processed/gemini__gemini-2.5-flash__aut_generations_success_only.pkl'),
 'success_csv': PosixPath('ai_data/processed/gemini__gemini-2.5-flash__aut_generations_success_only.csv')}

In [77]:
PROVIDER_MODELS = [
    ("openai", "gpt-5.4"),
    ("anthropic", "claude-sonnet-4-5"),
    ("gemini", "gemini-2.5-flash"),
]

all_model_dfs = []

for provider, model in PROVIDER_MODELS:
    df = load_all_generation_files(provider, model, AI_DATA_DIR)
    if df.empty:
        print(f"No files found for {provider} / {model}")
        continue

    df = deduplicate_generation_records(df)
    df = add_aut_validity_diagnostics(df)
    all_model_dfs.append(df)

combined_aut_df = pd.concat(all_model_dfs, ignore_index=True)

combined_aut_df["analysis_scenario_name"] = combined_aut_df["scenario_name"]

summary = (
    combined_aut_df
    .groupby(["provider", "model", "analysis_scenario_name", "temperature", "status"], dropna=False)
    .size()
    .reset_index(name="n")
    .sort_values(["provider", "model", "analysis_scenario_name", "temperature", "status"])
)

summary

,provider,model,analysis_scenario_name,temperature,status,n
0,anthropic,claude-sonnet-4-5,neutral_main_t1,1.0,success,250
1,anthropic,claude-sonnet-4-5,neutral_temperature_robustness,0.3,success,50
2,anthropic,claude-sonnet-4-5,neutral_temperature_robustness,0.7,success,50
3,anthropic,claude-sonnet-4-5,personality_grid,0.3,success,1600
4,anthropic,claude-sonnet-4-5,personality_grid,0.7,success,1600
5,anthropic,claude-sonnet-4-5,personality_grid,1.0,success,1600
6,gemini,gemini-2.5-flash,neutral_main_t1,1.0,success,250
7,gemini,gemini-2.5-flash,neutral_temperature_robustness,0.7,success,50
8,gemini,gemini-2.5-flash,neutral_temperature_robustness,1.3,success,50
9,gemini,gemini-2.5-flash,personality_grid,0.7,success,1600


In [78]:
success_counts = (
    combined_aut_df
    .query("status == 'success'")
    .groupby(["provider", "model", "analysis_scenario_name", "temperature"], dropna=False)
    .size()
    .reset_index(name="n")
    .sort_values(["provider", "model", "analysis_scenario_name", "temperature"])
)

success_counts

,provider,model,analysis_scenario_name,temperature,n
0,anthropic,claude-sonnet-4-5,neutral_main_t1,1.0,250
1,anthropic,claude-sonnet-4-5,neutral_temperature_robustness,0.3,50
2,anthropic,claude-sonnet-4-5,neutral_temperature_robustness,0.7,50
3,anthropic,claude-sonnet-4-5,personality_grid,0.3,1600
4,anthropic,claude-sonnet-4-5,personality_grid,0.7,1600
5,anthropic,claude-sonnet-4-5,personality_grid,1.0,1600
6,gemini,gemini-2.5-flash,neutral_main_t1,1.0,250
7,gemini,gemini-2.5-flash,neutral_temperature_robustness,0.7,50
8,gemini,gemini-2.5-flash,neutral_temperature_robustness,1.3,50
9,gemini,gemini-2.5-flash,personality_grid,0.7,1600


In [79]:
combined_aut_df = deduplicate_generation_records(combined_aut_df)
combined_aut_df = add_aut_validity_diagnostics(combined_aut_df)

combined_success_df = combined_aut_df.query("status == 'success'").copy()

combined_audit_pkl = CLEAN_AI_DIR / "all_models__aut_generations_all_records_with_errors.pkl"
combined_audit_csv = CLEAN_AI_DIR / "all_models__aut_generations_all_records_with_errors.csv"
combined_success_pkl = CLEAN_AI_DIR / "all_models__aut_generations_success_only.pkl"
combined_success_csv = CLEAN_AI_DIR / "all_models__aut_generations_success_only.csv"

combined_aut_df.to_pickle(combined_audit_pkl)
combined_aut_df.to_csv(combined_audit_csv, index=False)

combined_success_df.to_pickle(combined_success_pkl)
combined_success_df.to_csv(combined_success_csv, index=False)

print("Saved combined AUT files:")
print(combined_audit_pkl)
print(combined_audit_csv)
print(combined_success_pkl)
print(combined_success_csv)

Saved combined AUT files:
ai_data/processed/all_models__aut_generations_all_records_with_errors.pkl
ai_data/processed/all_models__aut_generations_all_records_with_errors.csv
ai_data/processed/all_models__aut_generations_success_only.pkl
ai_data/processed/all_models__aut_generations_success_only.csv


In [80]:
def show_aut_examples(
    df: pd.DataFrame,
    provider: Optional[str] = None,
    model: Optional[str] = None,
    condition_id: Optional[str] = None,
    scenario_name: Optional[str] = None,
    temperature: Optional[float] = None,
    n: int = 10,
    random_state: int = 42,
):
    use = df.copy()

    if provider is not None:
        use = use[use["provider"] == provider]
    if model is not None:
        use = use[use["model"] == model]
    if condition_id is not None:
        use = use[use["condition_id"] == condition_id]
    if scenario_name is not None:
        use = use[use["scenario_name"] == scenario_name]
    if temperature is not None:
        use = use[use["temperature"] == temperature]

    use = use[use["status"] == "success"].copy()

    if len(use) == 0:
        print("No matching successful generations.")
        return

    sample = use.sample(n=min(n, len(use)), random_state=random_state)

    for i, (_, row) in enumerate(sample.iterrows(), start=1):
        print("=" * 100)
        print(f"Example {i}")
        print(f"Provider/model: {row['provider']} / {row['model']}")
        print(f"Scenario: {row['scenario_name']}")
        print(f"Condition: {row['condition_id']} | Object: {row['object']}")
        print(f"Temperature: {row['temperature']}")
        print(f"Persona: {row.get('persona_id')}")
        print("-" * 100)
        print(row["text"])


show_aut_examples(
    combined_success_df,
    scenario_name="neutral_main_t1",
    condition_id="shoe",
    n=10,
)

Example 1
Provider/model: anthropic / claude-sonnet-4-5
Scenario: neutral_main_t1
Condition: shoe | Object: shoe
Temperature: 1.0
Persona: nan
----------------------------------------------------------------------------------------------------
A percussion instrument by tapping the heel and sole against hard surfaces to create rhythmic beats.
Example 2
Provider/model: openai / gpt-5.4
Scenario: neutral_main_t1
Condition: shoe | Object: shoe
Temperature: 1.0
Persona: nan
----------------------------------------------------------------------------------------------------
A shoe can serve as a doorstop to keep a door from swinging shut.
Example 3
Provider/model: openai / gpt-5.4
Scenario: neutral_main_t1
Condition: shoe | Object: shoe
Temperature: 1.0
Persona: nan
----------------------------------------------------------------------------------------------------
A shoe can be hung by its laces as a quirky hanging planter for small herbs or succulents.
Example 4
Provider/model: anthropic 